# Solar filament segmentation: tree-based candidate selection

The fixed post-processing rule (hysteresis thresholds + grouping) turns a probability map into instances. This notebook
tests a different last step on the **same** probability maps: build a tree of connected components over a ladder of
thresholds, train a scorer on the calibration images to predict each component's expected matched IoU, and pick a set
of disjoint components that maximises the expected micro PQ (exact dynamic programme, one global cut-off tau).

Protocol: the models are the v2 models trained on 457 images (no calibration or validation images). The scorer and tau are
fitted on the 111 calibration images only (scorer mode and tau by cross-fitting over capture dates). The 139 validation
images are scored once and compared with the v2 fixed rule on the very same maps (v2 validation PQ 0.4271).
The final scorer (calibration + validation) writes test CSVs for the held-out models and for the 707-image refit models.

**Nothing is submitted to Kaggle by this notebook.**

In [ ]:
import importlib.util, json, os, shutil, subprocess, sys, time
from pathlib import Path

if importlib.util.find_spec('pycocotools') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'pycocotools==2.0.11'])

import torch
INPUT_ROOT = Path(os.environ.get('FILAMENT_INPUT', '/kaggle/input'))
WORK_ROOT = Path(os.environ.get('FILAMENT_WORK', '/kaggle/working')).resolve()
os.chdir(WORK_ROOT)
Path('code/filament').mkdir(parents=True, exist_ok=True)
print('torch', torch.__version__, '| GPUs:', [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
      '| CPUs:', os.cpu_count())

In [ ]:
%%writefile code/filament/__init__.py
"""Solar filament segmentation package (written by the notebook)."""


In [ ]:
%%writefile code/filament/core.py
"""Image-only contrast segmentation and exact COCO-mask evaluation.

Only the training segmentation polygons supervise calibration. No spines,
chirality labels, external annotations, or test labels are consumed.
"""
import json
from pathlib import Path
from collections import defaultdict
import cv2
import numpy as np
from pycocotools import mask as coco
cv2.setNumThreads(2)


def load_annotations(data):
    a = json.loads((Path(data)/'train/MAGFiLO_1.0_Annotations_kaggle2026_train.json').read_text())
    records = defaultdict(list); annotations = defaultdict(list)
    for x in a['images']: records[x['file_name']].append(x)
    for x in a['annotations']: annotations[x['image_id']].append(x)
    return records, annotations


def ground_truth(records, annotations):
    result = []
    for record in records:
        masks = [coco.merge(coco.frPyObjects(a['segmentation'],record['height'],record['width']))
                 for a in annotations[record['id']]]
        result.append((record['id'], masks))
    return result


def contrast_maps(path):
    im = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if im is None: raise ValueError(f'Cannot read {path}')
    x = im.astype(np.float32)/255
    # Infer the bright solar disk from this image, not coordinates or filenames.
    _, disk = cv2.threshold(im,0,255,cv2.THRESH_BINARY+cv2.THRESH_OTSU)
    n, labels, stats, _ = cv2.connectedComponentsWithStats(disk,8)
    if n < 2: raise ValueError(f'No solar disk: {path}')
    mask = (labels == (1+np.argmax(stats[1:,cv2.CC_STAT_AREA]))).astype(np.uint8)
    contours,_ = cv2.findContours(mask,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
    mask[:] = 0; cv2.drawContours(mask,contours,-1,1,cv2.FILLED)
    mask = cv2.erode(mask,cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(25,25)))
    smooth = cv2.GaussianBlur(x,(0,0),1.0)
    # Local brightness deficit at two scales; denominator stabilizes limb contrast.
    maps=[]
    for sigma in (12.,32.,64.):
        background = cv2.GaussianBlur(smooth,(0,0),sigma)
        maps.append(np.clip((background-smooth)/np.maximum(background,.08),0,1))
    return im, mask, maps


def predict_from_maps(mask, maps, config):
    contrast = maps[config['scale']]
    binary = ((contrast > config['threshold']) & (mask>0)).astype(np.uint8)
    radius = config['closing_radius']
    if radius:
        k=cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(radius*2+1,radius*2+1))
        binary = cv2.morphologyEx(binary,cv2.MORPH_CLOSE,k)
    binary &= mask
    count, labels, stats,_ = cv2.connectedComponentsWithStats(binary,8)
    accepted=[]
    for lab in range(1,count):
        if stats[lab,cv2.CC_STAT_AREA] < config['min_area']: continue
        # Bound maximum component size to reject large image artifacts.
        if stats[lab,cv2.CC_STAT_AREA] > config.get('max_area',100000): continue
        elongation = config.get('min_elongation', 1.)
        if elongation > 1:
            x,y,w,h,_ = stats[lab]
            yy,xx=np.nonzero(labels[y:y+h,x:x+w]==lab)
            if len(xx)<3: continue
            eigen=np.linalg.eigvalsh(np.cov(np.stack([xx,yy])))
            ratio=np.sqrt((eigen[1]+1)/(eigen[0]+1))
            if ratio < elongation: continue
        accepted.append(lab)
    return labels,accepted


def encode_instances(labels, accepted):
    return [coco.encode(np.asfortranarray(labels==lab,dtype=np.uint8)) for lab in accepted]


def score_masks(gt, pred):
    ng,npred=len(gt),len(pred)
    if ng and npred:
        iou=coco.iou(pred,gt,[0]*ng).T
    else: iou=np.zeros((ng,npred),dtype=float)
    hit=iou > .5
    tp=int(hit.sum()); fp=int((hit.sum(axis=0)==0).sum()); fn=int((hit.sum(axis=1)==0).sum())
    total=float(iou[hit].sum())
    nonzero=iou[iou>0]
    return {'tp':tp,'fp':fp,'fn':fn,'sum_iou':total,
            'pq':total/(tp+.5*fp+.5*fn) if tp+.5*fp+.5*fn else 0.,
            'pair_iou':nonzero.tolist(),'pair_dice':(2*nonzero/(1+nonzero)).tolist(),
            'gt_overlap_degrees':(iou>0).sum(axis=1).tolist(),
            'pred_overlap_degrees':(iou>0).sum(axis=0).tolist()}


def aggregate(rows):
    sums={k:sum(r[k] for r in rows) for k in ('tp','fp','fn','sum_iou')}
    denom=sums['tp']+.5*sums['fp']+.5*sums['fn']
    sums['pq']=sums['sum_iou']/denom if denom else 0.
    return sums


In [ ]:
%%writefile code/filament/dl.py
"""Deep-learning extension: a soft-label U-Net trained on annotator polygons.

The classical baseline in ``core.py`` thresholds a local-contrast map. This module
replaces that pixel classifier with a ResNet34 U-Net trained only on the competition's
training polygons (no spines, chirality labels, external data, or test labels) and keeps
everything else identical: the date-grouped split in ``configs/split.json``, the micro-PQ
metric from ``core.score_masks``, and the CSV format written by ``run.predict``.

Design choices
--------------
* Native 2048 px resolution. Filaments are thin, so downscaling would cost IoU.
* Soft targets. Each image's target is the mean of its annotators' masks. The metric
  scores a prediction against every annotator set separately, so the consensus is the
  best single prediction to aim for.
* Hysteresis threshold, minimum area and limb erosion are tuned on the calibration
  subset only. The validation partition is evaluated once, at the end.
"""
import copy
import csv
import hashlib
import json
import time
from collections import defaultdict
from dataclasses import asdict, dataclass, replace
from pathlib import Path

import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from pycocotools import mask as coco
from scipy import ndimage

from .core import aggregate, encode_instances, ground_truth, load_annotations, score_masks


# --------------------------------------------------------------------------- configuration
@dataclass
class TrainConfig:
    """Training and inference settings for the U-Net."""

    crop: int = 512                 # training crop size in native pixels
    batch_size: int = 8
    epochs: int = 40
    steps_per_epoch: int = 150
    lr: float = 5e-4
    weight_decay: float = 1e-4
    positive_crop_prob: float = 0.7  # share of crops centred on an annotated pixel
    pretrained: bool = True          # ImageNet weights (needs internet; falls back to random)
    amp: bool = True                 # mixed precision on CUDA
    tta: bool = True                 # average over flips at inference
    workers: int = 3
    eval_every: int = 5              # epochs between quick tuning-set PQ checks
    eval_images: int = 40            # tuning images used for those quick checks
    seed: int = 2026
    encoder: str = "resnet34"        # see ENCODERS
    ema_decay: float = 0.0           # >0 keeps an exponential moving average of the weights


@dataclass
class PostConfig:
    """Turns a probability map into filament instances."""

    low: float = 0.5        # pixels above this form candidate regions
    high: float = 0.5       # a region is kept only if its peak probability reaches this
    min_area: int = 300
    max_area: int = 150000
    disk_erode: int = 10    # pixels removed from the solar limb
    closing_radius: int = 0
    group_dist: int = 0     # >0 merges fragments closer than about 2*group_dist px into one filament
    fill_holes: bool = False


def seed_everything(seed: int) -> None:
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


# --------------------------------------------------------------------------- data
def image_stats(img: np.ndarray) -> tuple:
    """Mean and standard deviation of the non-black pixels (the solar disk)."""
    pixels = img[img > 8].astype(np.float32)
    if pixels.size < 100:
        pixels = img.astype(np.float32).ravel()
    return float(pixels.mean()), float(pixels.std() + 1e-3)


def normalize(img: np.ndarray, mean: float, std: float) -> np.ndarray:
    return (img.astype(np.float32) - mean) / std


def solar_disk(img: np.ndarray) -> np.ndarray:
    """Filled solar-disk mask (uint8 0/1) from Otsu thresholding, as in core.contrast_maps."""
    _, bright = cv2.threshold(img, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    count, labels, stats, _ = cv2.connectedComponentsWithStats(bright, 8)
    if count < 2:
        return np.ones(img.shape, np.uint8)
    mask = (labels == 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])).astype(np.uint8)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    mask[:] = 0
    cv2.drawContours(mask, contours, -1, 1, cv2.FILLED)
    return mask


def erode_disk(mask: np.ndarray, radius: int) -> np.ndarray:
    if radius <= 0:
        return mask
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * radius + 1, 2 * radius + 1))
    return cv2.erode(mask, kernel)


def load_case(data_dir, name: str, records: dict, annotations: dict) -> dict:
    """Read one training image with its soft consensus target and per-annotator masks."""
    img = cv2.imread(str(Path(data_dir) / "train/train_images" / name), cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise ValueError(f"Cannot read training image {name}")
    gt = ground_truth(records[name], annotations)
    target = np.zeros(img.shape, np.float32)
    for _, masks in gt:
        if masks:
            target += coco.decode(coco.merge(masks))
    target /= max(1, len(gt))
    target_u8 = np.round(target * 255).astype(np.uint8)
    positives = np.flatnonzero(target_u8.ravel() >= 128)
    if positives.size > 4096:
        positives = np.random.default_rng(0).choice(positives, 4096, replace=False)
    mean, std = image_stats(img)
    return {"name": name, "image": img, "target": target_u8, "gt": gt, "disk": solar_disk(img),
            "mean": mean, "std": std, "positives": positives.astype(np.int64)}


def load_cases(data_dir, names, records, annotations, log=print) -> list:
    cases = []
    for index, name in enumerate(names, 1):
        cases.append(load_case(data_dir, name, records, annotations))
        if index % 50 == 0 or index == len(names):
            log(f"loaded {index}/{len(names)} images")
    return cases


def sample_crop(case: dict, size: int, rng: np.random.Generator, positive_prob: float):
    """Random crop, centred near an annotated pixel with probability ``positive_prob``."""
    img, target = case["image"], case["target"]
    height, width = img.shape
    if case["positives"].size and rng.random() < positive_prob:
        flat = int(case["positives"][rng.integers(case["positives"].size)])
        center_y, center_x = divmod(flat, width)
        y0 = center_y - int(rng.integers(size // 8, size - size // 8))
        x0 = center_x - int(rng.integers(size // 8, size - size // 8))
    else:
        y0 = int(rng.integers(0, max(1, height - size + 1)))
        x0 = int(rng.integers(0, max(1, width - size + 1)))
    y0 = int(np.clip(y0, 0, max(0, height - size)))
    x0 = int(np.clip(x0, 0, max(0, width - size)))
    patch = normalize(img[y0:y0 + size, x0:x0 + size], case["mean"], case["std"])
    soft = target[y0:y0 + size, x0:x0 + size].astype(np.float32) / 255.0
    if patch.shape != (size, size):
        pad = ((0, size - patch.shape[0]), (0, size - patch.shape[1]))
        patch = np.pad(patch, pad, constant_values=-case["mean"] / case["std"])
        soft = np.pad(soft, pad)
    return patch, soft


def augment(patch: np.ndarray, soft: np.ndarray, rng: np.random.Generator):
    """Dihedral flips plus brightness, contrast and noise jitter."""
    if rng.random() < 0.5:
        patch, soft = patch[:, ::-1], soft[:, ::-1]
    if rng.random() < 0.5:
        patch, soft = patch[::-1], soft[::-1]
    if rng.random() < 0.5:
        patch, soft = patch.T, soft.T
    patch = patch * rng.uniform(0.8, 1.25) + rng.uniform(-0.2, 0.2)
    patch = patch + rng.normal(0.0, rng.uniform(0.0, 0.05), patch.shape)
    return patch.astype(np.float32), soft


class CropDataset(torch.utils.data.Dataset):
    """Endless stream of random augmented crops; deterministic given (seed, epoch, index)."""

    def __init__(self, cases: list, cfg: TrainConfig, length: int):
        self.cases, self.cfg, self.length, self.epoch = cases, cfg, length, 0

    def set_epoch(self, epoch: int) -> None:
        self.epoch = epoch

    def __len__(self) -> int:
        return self.length

    def __getitem__(self, index: int):
        rng = np.random.default_rng([self.cfg.seed, self.epoch, index])
        case = self.cases[int(rng.integers(len(self.cases)))]
        patch, soft = sample_crop(case, self.cfg.crop, rng, self.cfg.positive_crop_prob)
        patch, soft = augment(patch, soft, rng)
        return (torch.from_numpy(np.ascontiguousarray(patch))[None],
                torch.from_numpy(np.ascontiguousarray(soft, dtype=np.float32))[None])


# --------------------------------------------------------------------------- model
def conv_block(c_in: int, c_out: int) -> nn.Sequential:
    return nn.Sequential(
        nn.Conv2d(c_in, c_out, 3, padding=1, bias=False), nn.BatchNorm2d(c_out), nn.ReLU(inplace=True),
        nn.Conv2d(c_out, c_out, 3, padding=1, bias=False), nn.BatchNorm2d(c_out), nn.ReLU(inplace=True),
    )


class UpBlock(nn.Module):
    """Upsample, concatenate the encoder skip connection, and convolve twice."""

    def __init__(self, c_in: int, c_skip: int, c_out: int):
        super().__init__()
        self.conv = conv_block(c_in + c_skip, c_out)

    def forward(self, x, skip):
        x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
        return self.conv(torch.cat([x, skip], dim=1))


# name -> (torchvision builder, weights enum, channels of stem, layer1, layer2, layer3, layer4)
ENCODERS = {
    "resnet34": ("resnet34", "ResNet34_Weights", (64, 64, 128, 256, 512)),
    "resnet50": ("resnet50", "ResNet50_Weights", (64, 256, 512, 1024, 2048)),
    "resnext50": ("resnext50_32x4d", "ResNeXt50_32X4D_Weights", (64, 256, 512, 1024, 2048)),
}


class FilamentUNet(nn.Module):
    """ResNet encoder, U-Net decoder, and a shallow branch at full input resolution.

    The ResNet stem halves the resolution immediately, which blurs thin barbs; the extra
    full-resolution branch gives the head direct access to the raw pixels. With the default
    ``encoder="resnet34"`` the layer names and shapes are those of the v1 model, so v1
    checkpoints load unchanged.
    """

    def __init__(self, pretrained: bool = True, encoder: str = "resnet34"):
        super().__init__()
        import torchvision

        builder_name, weights_name, (c0, c1, c2, c3, c4) = ENCODERS[encoder]
        builder = getattr(torchvision.models, builder_name)
        weights = getattr(torchvision.models, weights_name).DEFAULT if pretrained else None
        try:
            net = builder(weights=weights)
        except Exception as error:  # offline sessions cannot download the weights
            print(f"Pretrained weights unavailable ({type(error).__name__}); using random init.")
            net = builder(weights=None)
        conv1 = nn.Conv2d(1, 64, 7, stride=2, padding=3, bias=False)
        with torch.no_grad():
            conv1.weight.copy_(net.conv1.weight.sum(dim=1, keepdim=True))
        self.stem = nn.Sequential(conv1, net.bn1, net.relu)
        self.pool = net.maxpool
        self.layer1, self.layer2, self.layer3, self.layer4 = net.layer1, net.layer2, net.layer3, net.layer4
        self.up4 = UpBlock(c4, c3, 256)
        self.up3 = UpBlock(256, c2, 128)
        self.up2 = UpBlock(128, c1, 64)
        self.up1 = UpBlock(64, c0, 64)
        self.full = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1, bias=False), nn.BatchNorm2d(16), nn.ReLU(inplace=True),
            nn.Conv2d(16, 16, 3, padding=1, bias=False), nn.BatchNorm2d(16), nn.ReLU(inplace=True),
        )
        self.head = nn.Sequential(
            nn.Conv2d(64 + 16, 32, 3, padding=1, bias=False), nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.Conv2d(32, 1, 1),
        )

    def forward(self, x):
        height, width = x.shape[-2:]
        pad_h, pad_w = (-height) % 32, (-width) % 32
        if pad_h or pad_w:
            x = F.pad(x, (0, pad_w, 0, pad_h), mode="reflect")
        full = self.full(x)
        stem = self.stem(x)
        e1 = self.layer1(self.pool(stem))
        e2 = self.layer2(e1)
        e3 = self.layer3(e2)
        e4 = self.layer4(e3)
        d = self.up4(e4, e3)
        d = self.up3(d, e2)
        d = self.up2(d, e1)
        d = self.up1(d, stem)
        d = F.interpolate(d, size=x.shape[-2:], mode="bilinear", align_corners=False)
        return self.head(torch.cat([d, full], dim=1))[..., :height, :width]


def bce_dice_loss(logits: torch.Tensor, target: torch.Tensor, eps: float = 1.0) -> torch.Tensor:
    """Binary cross-entropy plus batch soft-Dice; both accept soft targets."""
    bce = F.binary_cross_entropy_with_logits(logits, target)
    prob = torch.sigmoid(logits)
    inter = (prob * target).sum()
    dice = 1.0 - (2.0 * inter + eps) / (prob.sum() + target.sum() + eps)
    return bce + dice


# --------------------------------------------------------------------------- inference
@torch.no_grad()
def predict_proba(model: nn.Module, img: np.ndarray, device: torch.device, cfg: TrainConfig,
                  tta=None) -> np.ndarray:
    """Probability map (float16, native resolution), optionally averaged over flips."""
    use_tta = cfg.tta if tta is None else tta
    use_amp = cfg.amp and device.type == "cuda"
    model.eval()
    mean, std = image_stats(img)
    x = torch.from_numpy(normalize(img, mean, std))[None, None].to(device)
    flips = [(), (3,), (2,), (2, 3)] if use_tta else [()]
    total = None
    for dims in flips:
        inp = torch.flip(x, dims) if dims else x
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            out = model(inp)
        out = torch.sigmoid(out.float())
        if dims:
            out = torch.flip(out, dims)
        total = out if total is None else total + out
    return (total / len(flips))[0, 0].cpu().numpy().astype(np.float16)


def fill_holes(binary: np.ndarray) -> np.ndarray:
    """Fill background regions not 4-connected to the border (= scipy binary_fill_holes), fast."""
    height, width = binary.shape
    flood = np.zeros((height + 2, width + 2), np.uint8)
    flood[1:-1, 1:-1] = binary > 0
    cv2.floodFill(flood, np.zeros((height + 4, width + 4), np.uint8), (0, 0), 1, flags=4)
    return ((binary > 0) | (flood[1:-1, 1:-1] == 0)).astype(np.uint8)


def segment(prob: np.ndarray, disk: np.ndarray, post: PostConfig):
    """Probability map -> (label image, accepted labels), the format encode_instances expects.

    Steps: threshold at ``low`` inside the (eroded) solar disk, optionally fill holes and
    close, label 8-connected regions, optionally merge regions closer than ~2*group_dist px
    (one filament is often annotated as several disjoint pieces), then keep instances whose
    area is in range and whose peak probability reaches ``high``. With ``group_dist=0`` and
    ``fill_holes=False`` this is exactly the v1 rule.
    """
    binary = ((prob > post.low) & (erode_disk(disk, post.disk_erode) > 0)).astype(np.uint8)
    if post.fill_holes:
        binary = fill_holes(binary)
    if post.closing_radius:
        size = post.closing_radius * 2 + 1
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (size, size))
        binary = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel)
    count, labels = cv2.connectedComponents(binary, connectivity=8, ltype=cv2.CV_32S)
    if post.group_dist > 0 and count > 2:
        size = 2 * post.group_dist + 1
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (size, size))
        count, grouped = cv2.connectedComponents(cv2.dilate(binary, kernel), connectivity=8, ltype=cv2.CV_32S)
        labels = np.where(binary > 0, grouped, 0).astype(np.int32)
    if count <= 1:
        return labels, []
    index = np.arange(1, count)
    areas = np.bincount(labels.ravel(), minlength=count)[1:]
    keep = (areas >= post.min_area) & (areas <= post.max_area)
    if post.high > post.low and keep.any():
        peaks = np.asarray(ndimage.maximum(prob.astype(np.float32), labels, index[keep]))
        keep[np.flatnonzero(keep)] = peaks >= post.high
    return labels, index[keep].tolist()


def encode_instances_fast(labels: np.ndarray, accepted: list) -> list:
    """Same RLEs as core.encode_instances, but reorders the label image to Fortran order once."""
    if not accepted:
        return []
    fortran = np.asfortranarray(labels)
    return [coco.encode((fortran == label).astype(np.uint8)) for label in accepted]


# --------------------------------------------------------------------------- evaluation
def count_matches(gt_masks: list, pred: list) -> dict:
    """tp/fp/fn/sum_iou with the same strict IoU > 0.5 rule as core.score_masks, but light."""
    n_gt, n_pred = len(gt_masks), len(pred)
    if not (n_gt and n_pred):
        return {"tp": 0, "fp": n_pred, "fn": n_gt, "sum_iou": 0.0}
    iou = coco.iou(pred, gt_masks, [0] * n_gt).T
    hit = iou > 0.5
    return {"tp": int(hit.sum()), "fp": int((hit.sum(axis=0) == 0).sum()),
            "fn": int((hit.sum(axis=1) == 0).sum()), "sum_iou": float(iou[hit].sum())}


def evaluate_probs(cases: list, probs: dict, post: PostConfig, full: bool = False) -> list:
    """One row per annotator record, exactly as run.evaluate builds them."""
    rows = []
    for case in cases:
        labels, accepted = segment(probs[case["name"]], case["disk"], post)
        pred = encode_instances(labels, accepted)
        for record_id, masks in case["gt"]:
            score = score_masks(masks, pred) if full else count_matches(masks, pred)
            rows.append({"file_name": case["name"], "image_record": record_id, **score})
    return rows


def bootstrap_ci(rows: list, samples: int = 2000, seed: int = 2026):
    """Capture-date cluster bootstrap of micro PQ, as in run.evaluate."""
    groups = defaultdict(list)
    for row in rows:
        groups[row["file_name"][:8]].append(row)
    clusters = [aggregate(group) for group in groups.values()]
    rng = np.random.default_rng(seed)
    pqs = [aggregate([clusters[i] for i in rng.integers(0, len(clusters), len(clusters))])["pq"]
           for _ in range(samples)]
    return np.percentile(pqs, [2.5, 97.5]).tolist(), len(groups)


def tune_post(cases: list, probs: dict, start: PostConfig, log=print) -> PostConfig:
    """Coordinate search over post-processing settings, using the tuning subset only."""
    best = start
    best_pq = aggregate(evaluate_probs(cases, probs, best))["pq"]
    log(f"post start pq={best_pq:.4f} {asdict(best)}")

    def trial(**changes):
        nonlocal best, best_pq
        candidate = replace(best, **changes)
        pq = aggregate(evaluate_probs(cases, probs, candidate))["pq"]
        log(f"post trial {changes} pq={pq:.4f}")
        if pq > best_pq:
            best, best_pq = candidate, pq

    for threshold in (0.3, 0.4, 0.5, 0.6):
        trial(low=threshold, high=threshold)
    for area in (50, 150, 300, 600):
        trial(min_area=area)
    for margin in (0.0, 0.15, 0.3):
        trial(high=min(0.95, best.low + margin))
    for radius in (0, 10, 20):
        trial(disk_erode=radius)
    log(f"post best pq={best_pq:.4f} {asdict(best)}")
    return best


# --------------------------------------------------------------------------- training
class ModelEMA:
    """Exponential moving average of all weights and BatchNorm statistics.

    The decay ramps up as (1 + step) / (10 + step) so early, random weights fade quickly.
    """

    def __init__(self, model: nn.Module, decay: float):
        self.decay, self.steps = decay, 0
        self.shadow = {k: v.detach().clone().float() if v.dtype.is_floating_point else v.detach().clone()
                       for k, v in model.state_dict().items()}

    @torch.no_grad()
    def update(self, model: nn.Module) -> None:
        self.steps += 1
        decay = min(self.decay, (1 + self.steps) / (10 + self.steps))
        for key, value in model.state_dict().items():
            if value.dtype.is_floating_point:
                self.shadow[key].mul_(decay).add_(value.detach().float(), alpha=1.0 - decay)
            else:
                self.shadow[key].copy_(value)

    def state_dict(self) -> dict:
        return {k: v.clone() for k, v in self.shadow.items()}


def train_model(model, train_cases, tune_cases, cfg: TrainConfig, device, out_dir, log=print) -> list:
    """Train with BCE+Dice, keep the checkpoint with the best quick tuning-set PQ.

    With ``cfg.ema_decay > 0`` the moving-average weights are the ones evaluated and saved.
    """
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    ema = ModelEMA(model, cfg.ema_decay) if cfg.ema_decay > 0 else None
    eval_model = copy.deepcopy(model) if ema else model
    use_amp = cfg.amp and device.type == "cuda"
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=cfg.lr, total_steps=cfg.epochs * cfg.steps_per_epoch, pct_start=0.1)
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    dataset = CropDataset(train_cases, cfg, cfg.steps_per_epoch * cfg.batch_size)
    eval_cases = tune_cases[: cfg.eval_images]
    best_pq, history = -1.0, []
    for epoch in range(cfg.epochs):
        dataset.set_epoch(epoch)
        loader = torch.utils.data.DataLoader(
            dataset, batch_size=cfg.batch_size, num_workers=cfg.workers,
            pin_memory=device.type == "cuda", drop_last=True)
        model.train()
        start, losses = time.time(), []
        for patch, soft in loader:
            patch, soft = patch.to(device, non_blocking=True), soft.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                logits = model(patch)
            loss = bce_dice_loss(logits.float(), soft)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            if ema:
                ema.update(model)
            losses.append(float(loss.detach()))
        record = {"epoch": epoch + 1, "loss": float(np.mean(losses)), "seconds": time.time() - start}
        if eval_cases and ((epoch + 1) % cfg.eval_every == 0 or epoch + 1 == cfg.epochs):
            if ema:
                eval_model.load_state_dict(ema.state_dict())
            probs = {c["name"]: predict_proba(eval_model, c["image"], device, cfg, tta=False) for c in eval_cases}
            record["tune_pq"] = aggregate(evaluate_probs(eval_cases, probs, PostConfig()))["pq"]
            if record["tune_pq"] > best_pq:
                best_pq = record["tune_pq"]
                torch.save(eval_model.state_dict(), out_dir / "best.pt")
        history.append(record)
        log(record)
    final = ema.state_dict() if ema else model.state_dict()
    torch.save(final, out_dir / "last.pt")
    if not (out_dir / "best.pt").exists():
        torch.save(final, out_dir / "best.pt")
    return history


# --------------------------------------------------------------------------- submission files
def list_images(directory) -> list:
    return sorted(p for p in Path(directory).iterdir() if p.suffix.lower() in (".jpg", ".jpeg"))


def write_submission(model, test_dir, post: PostConfig, cfg: TrainConfig, device, out_csv, log=print,
                     limit=None) -> dict:
    """Write one CSV row per predicted filament: ``filament_id,segmentation_rle`` (counts only)."""
    paths = list_images(test_dir)[:limit]
    manifest, start = [], time.time()
    with Path(out_csv).open("w", newline="") as handle:
        writer = csv.writer(handle)
        writer.writerow(["filament_id", "segmentation_rle"])
        for index, path in enumerate(paths, 1):
            tick = time.time()
            img = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
            labels, accepted = segment(predict_proba(model, img, device, cfg), solar_disk(img), post)
            instances = encode_instances(labels, accepted)
            for number, rle in enumerate(instances, 1):
                writer.writerow([f"{path.stem}_{number}", rle["counts"].decode("ascii")])
            manifest.append({"file_name": path.name, "instances": len(instances), "seconds": time.time() - tick})
            if index % 20 == 0:
                log(f"predicted {index}/{len(paths)}")
    return {"seconds": time.time() - start, "images": manifest}


def verify_submission(csv_path, test_dir) -> dict:
    """Decode and re-encode every mask; check headers, unique ids, and image coverage."""
    paths = {p.stem: p for p in list_images(test_dir)}
    sizes, seen, ids, count = {}, set(), set(), 0
    with Path(csv_path).open() as handle:
        reader = csv.DictReader(handle)
        if reader.fieldnames != ["filament_id", "segmentation_rle"]:
            raise ValueError(f"Unexpected CSV header: {reader.fieldnames}")
        for row in reader:
            fid = row["filament_id"]
            if fid in ids:
                raise ValueError(f"Duplicate filament_id {fid}")
            ids.add(fid)
            name, number = fid.rsplit("_", 1)
            if name not in paths or int(number) < 1:
                raise ValueError(f"Bad filament_id {fid}")
            if name not in sizes:
                sizes[name] = cv2.imread(str(paths[name]), cv2.IMREAD_GRAYSCALE).shape
            counts = row["segmentation_rle"]
            if '"' in counts or "'" in counts:
                raise ValueError(f"Quote character in RLE for {fid}")
            rle = {"size": list(sizes[name]), "counts": counts.encode("ascii")}
            mask = coco.decode(rle)
            if not mask.any():
                raise ValueError(f"Empty mask for {fid}")
            if coco.encode(np.asfortranarray(mask))["counts"] != rle["counts"]:
                raise ValueError(f"RLE is not canonical for {fid}")
            seen.add(name)
            count += 1
    digest = hashlib.sha256(Path(csv_path).read_bytes()).hexdigest()
    return {"status": "passed", "rows": count, "test_images": len(paths),
            "images_with_predictions": len(seen), "sha256": digest}


# --------------------------------------------------------------------------- experiment
def debug_overrides(cfg: TrainConfig) -> TrainConfig:
    """Tiny settings for a quick end-to-end check of the whole pipeline."""
    return replace(cfg, crop=256, batch_size=2, epochs=2, steps_per_epoch=4, workers=0,
                   eval_every=1, eval_images=2, tta=False, pretrained=False)


def run_experiment(data_dir, split: dict, out_dir, cfg=None, debug: bool = False,
                   baseline_pq: float = 0.1796, log=print) -> dict:
    """Train, tune on the calibration subset, evaluate once on validation, write a candidate CSV.

    Nothing is submitted to Kaggle. The candidate CSV and metrics are written to ``out_dir``.
    """
    cfg = debug_overrides(cfg or TrainConfig()) if debug else (cfg or TrainConfig())
    data_dir, out_dir = Path(data_dir), Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    seed_everything(cfg.seed)
    log(f"device={device} debug={debug}")

    calibration = set(split["calibration"])
    train_names = [n for n in split["training"] if n not in calibration]
    tune_names, val_names = list(split["calibration"]), list(split["validation"])
    test_dir = data_dir / "test/test_images"
    if debug:
        train_names, tune_names, val_names = train_names[:6], tune_names[:4], val_names[:4]
    log(f"train={len(train_names)} tune={len(tune_names)} validation={len(val_names)}")

    records, annotations = load_annotations(data_dir)
    train_cases = load_cases(data_dir, train_names, records, annotations, log)
    tune_cases = load_cases(data_dir, tune_names, records, annotations, log)

    model = FilamentUNet(cfg.pretrained, cfg.encoder).to(device)
    history = train_model(model, train_cases, tune_cases, cfg, device, out_dir, log)
    del train_cases
    model.load_state_dict(torch.load(out_dir / "best.pt", map_location=device))

    log("tuning post-processing on the calibration subset")
    tune_probs = {c["name"]: predict_proba(model, c["image"], device, cfg) for c in tune_cases}
    post = tune_post(tune_cases, tune_probs, PostConfig(), log)
    tune_pq = aggregate(evaluate_probs(tune_cases, tune_probs, post))["pq"]
    del tune_cases, tune_probs

    log("evaluating once on the held-out validation partition")
    val_cases = load_cases(data_dir, val_names, records, annotations, log)
    val_probs = {c["name"]: predict_proba(model, c["image"], device, cfg) for c in val_cases}
    rows = evaluate_probs(val_cases, val_probs, post, full=True)
    summary = aggregate(rows)
    interval, clusters = bootstrap_ci(rows)
    summary.update({"pq_bootstrap_95ci": interval, "images": len(val_cases),
                    "annotation_sets": len(rows), "date_clusters": clusters})
    log(f"validation PQ {summary['pq']:.4f} (baseline {baseline_pq:.4f}) 95% CI {interval}")

    log("writing the candidate submission (not uploaded anywhere)")
    csv_path = out_dir / "candidate_submission.csv"
    manifest = write_submission(model, test_dir, post, cfg, device, csv_path, log, limit=4 if debug else None)
    verification = verify_submission(csv_path, test_dir)
    metrics = {"validation": summary, "tuning_pq": tune_pq, "baseline_validation_pq": baseline_pq,
               "post": asdict(post), "train_config": asdict(cfg), "history": history,
               "verification": verification}
    (out_dir / "metrics.json").write_text(json.dumps(metrics, indent=2))
    (out_dir / "validation_rows.json").write_text(json.dumps(rows))
    (out_dir / "inference_manifest.json").write_text(json.dumps(manifest, indent=2))
    log(f"done: {verification}")
    return metrics


In [ ]:
%%writefile code/filament/suite.py
"""Experiment suite v2: several U-Nets trained in parallel, ensembled, tuned and compared.

Protocol (same partitions as v1, no leakage):
* every model trains on ``training`` minus ``calibration``;
* checkpoint choice, post-processing and the choice between candidates use ``calibration`` only;
* every candidate is then scored once on ``validation`` and gets its own test CSV.

One worker process per GPU trains (and later runs inference for) its share of the models, so a
Kaggle "GPU T4 x2" session trains two models at once. Probability maps are cached as 8-bit PNGs,
which makes ensembling and post-processing searches cheap and GPU-free.

Nothing is submitted to Kaggle by this module.

    python -m filament.suite train <job.json>     # worker entry points (used internally)
    python -m filament.suite infer <job.json>
"""
import json
import os
import subprocess
import sys
import time
from dataclasses import asdict, replace
from multiprocessing import get_context
from pathlib import Path

import cv2
import numpy as np
from pycocotools import mask as coco

from .core import aggregate, load_annotations
from .dl import (PostConfig, TrainConfig, bootstrap_ci, count_matches, encode_instances_fast, list_images, load_cases,
                 segment, solar_disk, verify_submission)

CODE_ROOT = Path(__file__).resolve().parents[1]

# Post-processing search space; ``margin`` sets high = min(0.95, low + margin).
SEARCH_SPACE = [
    ("low", [0.3, 0.4, 0.5, 0.6, 0.7]),
    ("margin", [0.0, 0.15, 0.3]),
    ("min_area", [100, 300, 600, 1000, 1600]),
    ("group_dist", [0, 4, 8, 16, 32]),
    ("fill_holes", [False, True]),
    ("disk_erode", [0, 10, 20]),
]


# --------------------------------------------------------------------------- suite definition
def default_suite(v1_weights=None, debug: bool = False) -> dict:
    """Two new models (one per T4), plus the v1 checkpoint if it is attached as an input."""
    base = TrainConfig(crop=768, epochs=32, steps_per_epoch=150, batch_size=8, workers=2,
                       ema_decay=0.998, eval_every=4)
    models = [
        {"name": "r34_c768_ema", "cfg": asdict(replace(base, encoder="resnet34", seed=2027)), "cost": 1.0},
        {"name": "r50_c768_ema", "cfg": asdict(replace(base, encoder="resnet50", seed=2028)), "cost": 1.6},
    ]
    if debug:
        for spec in models:
            spec["cfg"].update(crop=256, batch_size=2, epochs=2, steps_per_epoch=3, workers=0,
                               eval_every=1, eval_images=2, pretrained=False)
    external = []
    if v1_weights:
        external.append({"name": "v1_r34_c512", "encoder": "resnet34", "weights": str(v1_weights), "cost": 1.0})
    names = [m["name"] for m in models]
    ensembles = [names]
    if external:
        ensembles.append(names + [external[0]["name"]])
    return {"models": models, "external": external, "ensembles": ensembles}


def _candidates(suite: dict) -> dict:
    """Candidate name -> member model names: every single model, then every ensemble."""
    out = {m["name"]: [m["name"]] for m in suite["models"] + suite["external"]}
    for members in suite["ensembles"]:
        out["ens_" + "+".join(members)] = list(members)
    return out


# --------------------------------------------------------------------------- worker processes
def _log(*parts) -> None:
    print(time.strftime("%H:%M:%S"), *parts, flush=True)


def _device():
    import torch
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def _train_worker(job: dict) -> None:
    import torch
    from .dl import FilamentUNet, seed_everything, train_model

    device = _device()
    records, annotations = load_annotations(job["data"])
    for spec in job["models"]:
        cfg = TrainConfig(**spec["cfg"])
        out_dir = Path(job["out"]) / spec["name"]
        out_dir.mkdir(parents=True, exist_ok=True)
        _log(f"[{spec['name']}] device={device} gpu={torch.cuda.get_device_name(0) if device.type == 'cuda' else '-'}")
        seed_everything(cfg.seed)
        train_cases = load_cases(job["data"], job["train_names"], records, annotations, _log)
        for case in train_cases:  # training reads only image, target, mean/std and positives
            case.pop("disk", None)
            case.pop("gt", None)
        tune_cases = load_cases(job["data"], job["tune_names"][: cfg.eval_images], records, annotations, _log)
        model = FilamentUNet(cfg.pretrained, cfg.encoder).to(device)
        start = time.time()
        history = train_model(model, train_cases, tune_cases, cfg, device, out_dir, _log)
        (out_dir / "history.json").write_text(json.dumps(
            {"cfg": asdict(cfg), "history": history, "minutes": (time.time() - start) / 60}, indent=2))
        del train_cases, tune_cases, model
        torch.cuda.empty_cache()
        _log(f"[{spec['name']}] training finished")


def _infer_worker(job: dict) -> None:
    import torch
    from .dl import FilamentUNet, predict_proba

    device = _device()
    cfg = TrainConfig(tta=job.get("tta", True))
    for spec in job["models"]:
        model = FilamentUNet(False, spec["encoder"]).to(device)
        model.load_state_dict(torch.load(spec["weights"], map_location=device))
        out_dir = Path(job["probs"]) / spec["name"]
        out_dir.mkdir(parents=True, exist_ok=True)
        start, items = time.time(), sorted(job["images"].items())
        for index, (stem, path) in enumerate(items, 1):
            target = out_dir / f"{stem}.png"
            if target.exists():
                continue
            img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
            prob = predict_proba(model, img, device, cfg).astype(np.float32)
            cv2.imwrite(str(target), np.round(prob * 255).astype(np.uint8))
            if index % 50 == 0 or index == len(items):
                _log(f"[{spec['name']}] inference {index}/{len(items)}")
        _log(f"[{spec['name']}] inference finished in {(time.time() - start) / 60:.1f} min")
        del model
        torch.cuda.empty_cache()


def _assign(items: list, slots: int) -> list:
    """Greedy longest-first assignment of weighted jobs to GPU slots."""
    bins = [[] for _ in range(slots)]
    loads = [0.0] * slots
    for item in sorted(items, key=lambda i: -i.get("cost", 1.0)):
        k = int(np.argmin(loads))
        bins[k].append(item)
        loads[k] += item.get("cost", 1.0)
    return bins


def _gpu_slots() -> list:
    import torch
    count = torch.cuda.device_count() if torch.cuda.is_available() else 0
    return list(range(count)) or [None]


def _tail(path: Path, lines: int = 1) -> str:
    try:
        text = path.read_text(errors="replace").strip().splitlines()
    except FileNotFoundError:
        return "(no output yet)"
    return "\n".join(text[-lines:]) if text else "(no output yet)"


def _launch(kind: str, job_items: list, common: dict, work: Path) -> list:
    """Start one worker per GPU; returns (process, log path, file handle) triples."""
    slots = _gpu_slots()
    procs = []
    for slot, (gpu, items) in enumerate(zip(slots, _assign(job_items, len(slots)))):
        if not items:
            continue
        job_path = work / f"{kind}_job{slot}.json"
        job_path.write_text(json.dumps({**common, "models": items}))
        log_path = work / f"{kind}_worker{slot}.log"
        env = dict(os.environ, PYTHONUNBUFFERED="1",
                   CUDA_VISIBLE_DEVICES="" if gpu is None else str(gpu),
                   PYTHONPATH=os.pathsep.join(filter(None, [str(CODE_ROOT), os.environ.get("PYTHONPATH")])))
        handle = log_path.open("w")
        proc = subprocess.Popen([sys.executable, "-u", "-m", "filament.suite", kind, str(job_path)],
                                stdout=handle, stderr=subprocess.STDOUT, env=env)
        procs.append((proc, log_path, handle))
        _log(f"started {kind} worker {slot} on gpu {gpu}: {[i['name'] for i in items]}")
    return procs


def _wait(procs: list, every: float = 300.0) -> None:
    last = time.time()
    while any(p.poll() is None for p, _, _ in procs):
        time.sleep(10)
        if time.time() - last >= every:
            last = time.time()
            for _, log_path, _ in procs:
                _log(f"{log_path.name}: {_tail(log_path)}")
    for proc, log_path, handle in procs:
        handle.close()
        if proc.returncode != 0:
            raise RuntimeError(f"{log_path.name} exited with {proc.returncode}:\n{_tail(log_path, 40)}")


# --------------------------------------------------------------------------- parallel evaluation
_STATE: dict = {}


def _eval_image(args):
    index, post = args
    case = _STATE["cases"][index]
    labels, accepted = segment(_STATE["probs"][case["name"]], case["disk"], PostConfig(**post))
    pred = encode_instances_fast(labels, accepted)
    return [{"file_name": case["name"], "image_record": rid, **count_matches(masks, pred)}
            for rid, masks in case["gt"]]


class Evaluator:
    """Scores PostConfigs on a fixed set of cases and probability maps, in parallel, with a cache."""

    def __init__(self, cases: list, probs: dict, workers: int):
        _STATE.update(cases=cases, probs=probs)
        self.n, self.cache = len(cases), {}
        self.pool = get_context("fork").Pool(workers) if workers > 1 else None

    def rows(self, post: PostConfig) -> list:
        key = tuple(sorted(asdict(post).items()))
        if key not in self.cache:
            args = [(i, asdict(post)) for i in range(self.n)]
            parts = self.pool.map(_eval_image, args, chunksize=2) if self.pool else map(_eval_image, args)
            self.cache[key] = [row for part in parts for row in part]
        return self.cache[key]

    def pq(self, post: PostConfig) -> float:
        return aggregate(self.rows(post))["pq"]

    def close(self) -> None:
        if self.pool:
            self.pool.close()
            self.pool.join()
        _STATE.clear()


def _make_post(params: dict) -> PostConfig:
    return PostConfig(low=params["low"], high=min(0.95, params["low"] + params["margin"]),
                      min_area=params["min_area"], group_dist=params["group_dist"],
                      fill_holes=params["fill_holes"], disk_erode=params["disk_erode"])


def tune_post_v2(evaluator: Evaluator, passes: int = 2, log=_log):
    """Coordinate search over SEARCH_SPACE (calibration images only); returns (PostConfig, pq)."""
    params = {"low": 0.5, "margin": 0.0, "min_area": 300, "group_dist": 0, "fill_holes": False, "disk_erode": 10}
    best = evaluator.pq(_make_post(params))
    log(f"  start pq={best:.4f} {params}")
    for round_index in range(passes):
        improved = False
        for key, values in SEARCH_SPACE:
            for value in values:
                if value == params[key]:
                    continue
                trial = dict(params, **{key: value})
                score = evaluator.pq(_make_post(trial))
                if score > best + 1e-9:
                    params, best, improved = trial, score, True
        log(f"  pass {round_index + 1}: pq={best:.4f} {params}")
        if not improved:
            break
    return _make_post(params), best


# --------------------------------------------------------------------------- probability maps
def _load_probs(probs_dir: Path, members: list, names: list) -> dict:
    out = {}
    for name in names:
        total = None
        for member in members:
            path = probs_dir / member / f"{Path(name).stem}.png"
            arr = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
            if arr is None:
                raise FileNotFoundError(path)
            total = arr.astype(np.float32) if total is None else total + arr
        out[name] = (total / (255.0 * len(members))).astype(np.float16)
    return out


def _csv_image(args):
    path, members, probs_dir, post = args
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    prob = _load_probs(Path(probs_dir), members, [Path(path).name])[Path(path).name]
    labels, accepted = segment(prob, solar_disk(img), PostConfig(**post))
    return [(f"{Path(path).stem}_{n}", rle["counts"].decode("ascii"))
            for n, rle in enumerate(encode_instances_fast(labels, accepted), 1)]


def write_candidate_csv(test_paths: list, members: list, probs_dir: Path, post: PostConfig, out_csv: Path,
                        workers: int) -> None:
    args = [(str(p), members, str(probs_dir), asdict(post)) for p in test_paths]
    if workers > 1:
        with get_context("fork").Pool(workers) as pool:
            parts = pool.map(_csv_image, args, chunksize=2)
    else:
        parts = list(map(_csv_image, args))
    with out_csv.open("w") as handle:
        handle.write("filament_id,segmentation_rle\n")
        for part in parts:
            for fid, counts in part:
                handle.write(f"{fid},{counts}\n")


# --------------------------------------------------------------------------- diagnostics
def diagnostics(names: list, records: dict, annotations: dict, tune_cases: list, val_cases: list,
                workers: int, log=_log) -> dict:
    """What limits the score: annotator agreement, multi-part filaments, instance formation."""
    out = {}
    parts = [len(a["segmentation"]) if isinstance(a["segmentation"], list) else 1
             for name in names for record in records[name] for a in annotations[record["id"]]]
    out["annotations"] = len(parts)
    out["share_multi_polygon"] = float(np.mean([p > 1 for p in parts])) if parts else 0.0
    components = []
    for case in tune_cases + val_cases:
        for _, masks in case["gt"]:
            for rle in masks:
                x, y, w, h = [int(v) for v in coco.toBbox(rle)]
                crop = coco.decode(rle)[y:y + h + 1, x:x + w + 1]
                components.append(cv2.connectedComponents(crop, connectivity=8)[0] - 1)
    out["gt_instances_scored"] = len(components)
    out["share_gt_instances_multi_part"] = float(np.mean([c > 1 for c in components])) if components else 0.0
    pairs, multi = [], 0
    for case in tune_cases + val_cases:
        sets = case["gt"]
        multi += len(sets) > 1
        for i, (_, reference) in enumerate(sets):
            for j, (_, other) in enumerate(sets):
                if i != j:
                    pairs.append(count_matches(reference, other))
    out["images_with_2plus_annotators"] = multi
    out["human_agreement"] = aggregate(pairs) if pairs else None
    probs = {c["name"]: (c["target"].astype(np.float32) / 255.0).astype(np.float16) for c in tune_cases}
    evaluator = Evaluator(tune_cases, probs, workers)
    try:
        for group in (0, 8, 16, 32):
            post = PostConfig(low=0.49, high=0.49, min_area=1, disk_erode=0, group_dist=group)
            out[f"consensus_as_prediction_pq_group{group}"] = evaluator.pq(post)
    finally:
        evaluator.close()
    log(f"diagnostics: {json.dumps(out)}")
    return out


# --------------------------------------------------------------------------- orchestration
def run_suite(data_dir, split: dict, out_dir, suite: dict, debug: bool = False, baseline_pq: float = 0.1796,
              v1_val_pq: float = 0.4111, probs_root=None, workers=None, log=_log) -> dict:
    """Train, infer, ensemble, tune on calibration, score validation, write one CSV per candidate."""
    cv2.setNumThreads(1)  # keep OpenCV's thread pool idle in this process: it forks evaluation workers
    data_dir, out_dir = Path(data_dir), Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    work = out_dir / "work"
    work.mkdir(exist_ok=True)
    probs_dir = Path(probs_root or "/tmp/filament_probs") / out_dir.name
    workers = workers or max(1, min(4, os.cpu_count() or 1))
    timings, t0 = {}, time.time()

    calibration = set(split["calibration"])
    train_names = [n for n in split["training"] if n not in calibration]
    tune_names, val_names = list(split["calibration"]), list(split["validation"])
    test_paths = list_images(data_dir / "test/test_images")
    if debug:
        train_names, tune_names, val_names, test_paths = train_names[:6], tune_names[:4], val_names[:4], test_paths[:4]
    log(f"suite: train={len(train_names)} calibration={len(tune_names)} validation={len(val_names)} "
        f"test={len(test_paths)} gpus={_gpu_slots()} cpu_workers={workers}")

    # 1. training, one worker per GPU; diagnostics and case loading overlap with it on the CPU
    procs = _launch("train", suite["models"],
                    {"data": str(data_dir), "out": str(out_dir / "models"),
                     "train_names": train_names, "tune_names": tune_names}, work)
    records, annotations = load_annotations(data_dir)
    tune_cases = load_cases(data_dir, tune_names, records, annotations, log)
    val_cases = load_cases(data_dir, val_names, records, annotations, log)
    diag = diagnostics(train_names + tune_names + val_names, records, annotations, tune_cases, val_cases,
                       workers, log)
    _wait(procs)
    timings["train_min"] = (time.time() - t0) / 60
    histories = {m["name"]: json.loads((out_dir / "models" / m["name"] / "history.json").read_text())
                 for m in suite["models"]}

    # 2. inference: cached probability maps for calibration, validation and test images
    t1 = time.time()
    images = {Path(n).stem: str(data_dir / "train/train_images" / n) for n in tune_names + val_names}
    images.update({p.stem: str(p) for p in test_paths})
    infer_models = [{"name": m["name"], "encoder": m["cfg"]["encoder"], "cost": m.get("cost", 1.0),
                     "weights": str(out_dir / "models" / m["name"] / "best.pt")} for m in suite["models"]]
    infer_models += suite["external"]
    _wait(_launch("infer", infer_models, {"probs": str(probs_dir), "images": images, "tta": True}, work))
    timings["infer_min"] = (time.time() - t1) / 60

    # 3. candidates: tune on calibration, choose on calibration, score every candidate on validation
    t2 = time.time()
    results = {}
    for cand, members in _candidates(suite).items():
        log(f"candidate {cand}")
        evaluator = Evaluator(tune_cases, _load_probs(probs_dir, members, tune_names), workers)
        try:
            post, tune_pq = tune_post_v2(evaluator, passes=1 if debug else 2, log=log)
        finally:
            evaluator.close()
        evaluator = Evaluator(val_cases, _load_probs(probs_dir, members, val_names), workers)
        try:
            rows = evaluator.rows(post)
            baseline_rows = evaluator.rows(replace(post, group_dist=0, fill_holes=False))
        finally:
            evaluator.close()
        summary = aggregate(rows)
        interval, clusters = bootstrap_ci(rows)
        summary.update(pq_bootstrap_95ci=interval, date_clusters=clusters, annotation_sets=len(rows))
        results[cand] = {"members": members, "post": asdict(post), "calibration_pq": tune_pq,
                         "validation": summary,
                         "validation_pq_without_grouping": aggregate(baseline_rows)["pq"]}
        log(f"  calibration pq={tune_pq:.4f}  validation pq={summary['pq']:.4f} "
            f"CI {interval[0]:.4f}-{interval[1]:.4f}  tp/fp/fn={summary['tp']}/{summary['fp']}/{summary['fn']}")
    timings["tune_min"] = (time.time() - t2) / 60

    # 4. one candidate CSV per candidate (never uploaded by this code)
    t3 = time.time()
    selected = max(results, key=lambda c: (results[c]["calibration_pq"], -len(results[c]["members"])))
    for cand, result in results.items():
        cand_dir = out_dir / "candidates" / cand
        cand_dir.mkdir(parents=True, exist_ok=True)
        csv_path = cand_dir / "submission.csv"
        write_candidate_csv(test_paths, result["members"], probs_dir, PostConfig(**result["post"]), csv_path, workers)
        result["csv"] = verify_submission(csv_path, data_dir / "test/test_images")
        if debug:
            result["csv"]["note"] = "debug run: only a few test images"
        log(f"{cand}: {result['csv']}")
    (out_dir / "SELECTED_BY_CALIBRATION.txt").write_text(
        f"{selected}\n{results[selected]['csv']['sha256']}\n")
    timings["csv_min"] = (time.time() - t3) / 60
    timings["total_min"] = (time.time() - t0) / 60

    metrics = {"selected_by_calibration": selected, "candidates": results, "diagnostics": diag,
               "histories": histories, "suite": suite, "timings": timings,
               "reference": {"classical_baseline_validation_pq": baseline_pq, "v1_validation_pq": v1_val_pq},
               "debug": debug}
    (out_dir / "suite_metrics.json").write_text(json.dumps(metrics, indent=2))
    log(f"selected by calibration: {selected}; timings {timings}")
    return metrics


def run_refit(data_dir, split: dict, out_dir, models: list, post: dict, external=None, debug: bool = False,
              probs_root=None, workers=None, log=_log) -> dict:
    """Retrain ``models`` on every labelled image (training + validation) and write one test CSV.

    Use this only after a suite run has chosen the configuration and ``post`` on calibration data.
    There is no held-out data left, so the checkpoint is the final (EMA) weights and the reported
    estimate of quality remains the suite's validation score. Nothing is submitted to Kaggle.
    """
    cv2.setNumThreads(1)
    data_dir, out_dir = Path(data_dir), Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    work = out_dir / "work"
    work.mkdir(exist_ok=True)
    probs_dir = Path(probs_root or "/tmp/filament_probs") / out_dir.name
    workers = workers or max(1, min(4, os.cpu_count() or 1))
    external = list(external or [])
    train_names = list(split["training"]) + list(split["validation"])
    test_paths = list_images(data_dir / "test/test_images")
    if debug:
        train_names, test_paths = train_names[:6], test_paths[:4]
    log(f"refit: train={len(train_names)} test={len(test_paths)} models={[m['name'] for m in models]} "
        f"external={[e['name'] for e in external]} post={post}")
    t0 = time.time()
    _wait(_launch("train", models, {"data": str(data_dir), "out": str(out_dir / "models"),
                                    "train_names": train_names, "tune_names": []}, work))
    infer_models = [{"name": m["name"], "encoder": m["cfg"]["encoder"], "cost": m.get("cost", 1.0),
                     "weights": str(out_dir / "models" / m["name"] / "last.pt")} for m in models] + external
    _wait(_launch("infer", infer_models, {"probs": str(probs_dir), "tta": True,
                                          "images": {p.stem: str(p) for p in test_paths}}, work))
    members = [m["name"] for m in infer_models]
    csv_path = out_dir / "submission.csv"
    write_candidate_csv(test_paths, members, probs_dir, PostConfig(**post), csv_path, workers)
    verification = verify_submission(csv_path, data_dir / "test/test_images")
    metrics = {"members": members, "post": post, "train_images": len(train_names), "csv": verification,
               "histories": {m["name"]: json.loads((out_dir / "models" / m["name"] / "history.json").read_text())
                             for m in models},
               "minutes": (time.time() - t0) / 60, "debug": debug}
    (out_dir / "refit_metrics.json").write_text(json.dumps(metrics, indent=2))
    log(f"refit done: {verification}")
    return metrics


def compare_submissions(reference_csv, other_csv, test_dir) -> dict:
    """Agreement between two submission files on the test images (no labels involved).

    ``matched_pq`` scores ``other`` against ``reference`` with the competition's strict IoU > 0.5
    matching; ``mean_pixel_dice`` compares the union masks image by image. A sanity check that a
    retrained model behaves like the validated one, not a measure of accuracy.
    """
    import csv as _csv
    from collections import defaultdict

    paths = {p.stem: p for p in list_images(test_dir)}
    shapes = {}

    def read(path):
        out = defaultdict(list)
        with Path(path).open() as handle:
            for row in _csv.DictReader(handle):
                stem = row["filament_id"].rsplit("_", 1)[0]
                if stem not in shapes:
                    shapes[stem] = cv2.imread(str(paths[stem]), cv2.IMREAD_GRAYSCALE).shape
                out[stem].append({"size": list(shapes[stem]), "counts": row["segmentation_rle"].encode("ascii")})
        return out

    ref, other = read(reference_csv), read(other_csv)
    rows, dices = [], []
    for stem in sorted(set(ref) | set(other)):
        a, b = ref.get(stem, []), other.get(stem, [])
        rows.append(count_matches(a, b))
        area_a = float(coco.area(coco.merge(a))) if a else 0.0
        area_b = float(coco.area(coco.merge(b))) if b else 0.0
        if area_a + area_b == 0:
            continue
        inter = float(coco.area(coco.merge([coco.merge(a), coco.merge(b)], intersect=True))) if a and b else 0.0
        dices.append(2 * inter / (area_a + area_b))
    summary = aggregate(rows)
    return {"reference_instances": sum(map(len, ref.values())), "other_instances": sum(map(len, other.values())),
            "images_compared": len(rows), "matched_pq": summary["pq"], "tp": summary["tp"], "fp": summary["fp"],
            "fn": summary["fn"], "mean_pixel_dice": float(np.mean(dices)) if dices else 1.0}


def summary_table(metrics: dict) -> str:
    lines = [f"{'candidate':42s} {'calib PQ':>9s} {'val PQ':>8s} {'95% CI':>17s} {'TP/FP/FN':>16s} {'no-group':>9s}"]
    for cand, r in sorted(metrics["candidates"].items(), key=lambda kv: -kv[1]["calibration_pq"]):
        v = r["validation"]
        mark = " <- selected" if cand == metrics["selected_by_calibration"] else ""
        lines.append(f"{cand:42s} {r['calibration_pq']:9.4f} {v['pq']:8.4f} "
                     f"{v['pq_bootstrap_95ci'][0]:8.4f}-{v['pq_bootstrap_95ci'][1]:.4f} "
                     f"{v['tp']:>5d}/{v['fp']}/{v['fn']:<5d} {r['validation_pq_without_grouping']:9.4f}{mark}")
    return "\n".join(lines)


def main(argv=None) -> None:
    argv = argv or sys.argv[1:]
    kind, job_path = argv[0], argv[1]
    job = json.loads(Path(job_path).read_text())
    {"train": _train_worker, "infer": _infer_worker}[kind](job)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile code/filament/treesel.py
"""Candidate selection on a component tree (replaces a single fixed threshold).

A fixed rule (one threshold, hysteresis, a minimum area) picks one segmentation of a probability
map. Here every connected component at a ladder of thresholds is a *candidate* (a node of a tree:
raising the threshold splits or shrinks a component). A scorer predicts, from the shape and the
probabilities of a candidate, the expected matched IoU ``u`` it would earn against a random
annotator record. Because pooled PQ is ``sum(matched IoU) / (TP + FP/2 + FN/2)``, a prediction
adds to the numerator ``u`` on average and to the denominator ``1/2`` per record, so the best
submission maximises ``sum(u - tau)`` over a set of disjoint candidates, with ``tau`` about
half the achievable PQ. The best disjoint set of a tree (an antichain) is found exactly by dynamic
programming. ``tau`` is tuned on calibration images only.

Only image pixels (through the probability map) and the solar disk feed the scorer; ground truth
is used to train it on calibration images, never at test time.
"""
from __future__ import annotations

from dataclasses import asdict, dataclass, field

import cv2
import numpy as np
from pycocotools import mask as coco

from .dl import erode_disk, fill_holes

LEVELS = (0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.70, 0.80)


@dataclass
class TreeConfig:
    levels: tuple = LEVELS
    min_area: int = 150
    max_area: int = 150000
    disk_erode: int = 10
    fill_holes: bool = True
    ring: int = 10


# --------------------------------------------------------------------------- the tree
def build_tree(prob: np.ndarray, disk: np.ndarray, cfg: TreeConfig) -> dict:
    """Components of ``prob > level`` inside the eroded disk, for every level, linked by inclusion.

    A component identical to its parent (same area) is not a new node; the parent just remembers
    the highest level at which it persists. Returns node arrays plus the label image of each level.
    """
    prob32 = prob.astype(np.float32)
    valid = erode_disk(disk, cfg.disk_erode) > 0
    nodes, node_of, label_images = [], {}, []
    for li, threshold in enumerate(cfg.levels):
        binary = ((prob32 > threshold) & valid).astype(np.uint8)
        if cfg.fill_holes:
            binary = fill_holes(binary)
        count, labels, stats, _ = cv2.connectedComponentsWithStats(binary, connectivity=8, ltype=cv2.CV_32S)
        label_images.append(labels)
        for k in range(1, count):
            x, y, w, h, area = (int(v) for v in stats[k])
            if area < cfg.min_area:
                continue
            parent = -1
            if li > 0:
                inside = labels[y:y + h, x:x + w] == k
                py, px = np.unravel_index(int(np.argmax(inside)), inside.shape)
                parent = node_of.get((li - 1, int(label_images[li - 1][y + py, x + px])), -1)
            if parent >= 0 and nodes[parent]["area"] == area:
                node_of[(li, k)] = parent
                nodes[parent]["last_level"] = li
                continue
            node_of[(li, k)] = len(nodes)
            nodes.append({"level": li, "label": k, "x": x, "y": y, "w": w, "h": h, "area": area,
                          "parent": parent, "last_level": li})
    return {"nodes": nodes, "labels": label_images, "valid": valid, "prob": prob32, "cfg": cfg}


def node_mask(tree: dict, index: int) -> np.ndarray:
    """Boolean mask of one node, cropped to its bounding box."""
    node = tree["nodes"][index]
    crop = tree["labels"][node["level"]][node["y"]:node["y"] + node["h"], node["x"]:node["x"] + node["w"]]
    return crop == node["label"]


FEATURE_NAMES = [
    "t_low", "t_high", "persistence", "area", "log_area", "bbox_long", "bbox_short", "extent",
    "elongation", "major", "minor", "solidity", "compactness",
    "p_mean", "p_max", "p_std", "p_q25", "p_q75", "frac_gt50", "ring_mean", "contrast",
    "area_ratio_parent", "n_children", "n_descendants", "max_child_ratio", "sum_child_ratio", "depth",
    "rho", "disk_activity", "disk_mean_p", "n_nodes_at_half",
]


def node_features(tree: dict, disk: np.ndarray) -> np.ndarray:
    """(N, len(FEATURE_NAMES)) float32 descriptors of every node; image pixels and the disk only."""
    nodes, cfg, prob, valid = tree["nodes"], tree["cfg"], tree["prob"], tree["valid"]
    n = len(nodes)
    if n == 0:
        return np.zeros((0, len(FEATURE_NAMES)), np.float32)
    levels = np.asarray(cfg.levels, np.float32)
    parent = np.array([nd["parent"] for nd in nodes])
    area = np.array([nd["area"] for nd in nodes], np.float64)
    children = [[] for _ in range(n)]
    for i, p in enumerate(parent):
        if p >= 0:
            children[p].append(i)
    n_desc = np.zeros(n)
    depth = np.zeros(n)
    for i in range(n):                      # parents precede children
        if parent[i] >= 0:
            depth[i] = depth[parent[i]] + 1
    for i in range(n - 1, -1, -1):
        n_desc[i] = sum(1 + n_desc[c] for c in children[i])
    moments = cv2.moments(disk.astype(np.uint8), binaryImage=True)
    disk_area = max(moments["m00"], 1.0)
    cx, cy = moments["m10"] / disk_area, moments["m01"] / disk_area
    radius = np.sqrt(disk_area / np.pi)
    valid_area = max(float(valid.sum()), 1.0)
    activity = float(((prob > 0.5) & valid).sum()) / valid_area
    mean_p = float(prob[valid].mean()) if valid.any() else 0.0
    half = float(levels[np.argmin(np.abs(levels - 0.5))])
    at_half = sum(1 for nd in nodes if levels[nd["level"]] <= half <= levels[nd["last_level"]])
    height, width = prob.shape
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * cfg.ring + 1, 2 * cfg.ring + 1))
    out = np.zeros((n, len(FEATURE_NAMES)), np.float32)
    for i, nd in enumerate(nodes):
        mask = node_mask(tree, i)
        x, y, w, h = nd["x"], nd["y"], nd["w"], nd["h"]
        values = prob[y:y + h, x:x + w][mask]
        m = cv2.moments(mask.astype(np.uint8), binaryImage=True)
        m00 = max(m["m00"], 1.0)
        mu20, mu02, mu11 = m["mu20"] / m00, m["mu02"] / m00, m["mu11"] / m00
        trace, det = mu20 + mu02, mu20 * mu02 - mu11 ** 2
        disc = np.sqrt(max(trace ** 2 / 4 - det, 0.0))
        l1, l2 = trace / 2 + disc, max(trace / 2 - disc, 1e-3)
        contours, _ = cv2.findContours(mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        perimeter = sum(cv2.arcLength(c, True) for c in contours)
        hull = cv2.contourArea(cv2.convexHull(np.vstack(contours))) if contours else nd["area"]
        y0, y1, x0, x1 = max(0, y - cfg.ring), min(height, y + h + cfg.ring), max(0, x - cfg.ring), min(width, x + w + cfg.ring)
        window = np.zeros((y1 - y0, x1 - x0), np.uint8)
        window[y - y0:y - y0 + h, x - x0:x - x0 + w] = mask
        ring = (cv2.dilate(window, kernel) > 0) & (window == 0) & valid[y0:y1, x0:x1]
        ring_mean = float(prob[y0:y1, x0:x1][ring].mean()) if ring.any() else 0.0
        kids = area[children[i]]
        centre = np.hypot(x + w / 2 - cx, y + h / 2 - cy) / radius
        out[i] = [
            levels[nd["level"]], levels[nd["last_level"]], levels[nd["last_level"]] - levels[nd["level"]],
            nd["area"], np.log1p(nd["area"]), max(w, h), min(w, h), nd["area"] / (w * h),
            np.sqrt(l1 / l2), 4 * np.sqrt(l1), 4 * np.sqrt(l2), nd["area"] / max(hull, 1.0),
            4 * np.pi * nd["area"] / max(perimeter, 1.0) ** 2,
            values.mean(), values.max(), values.std(), np.percentile(values, 25), np.percentile(values, 75),
            float((values > 0.5).mean()), ring_mean, float(values.mean()) - ring_mean,
            nd["area"] / area[parent[i]] if parent[i] >= 0 else 1.0, len(kids), n_desc[i],
            kids.max() / nd["area"] if len(kids) else 0.0, kids.sum() / nd["area"] if len(kids) else 0.0, depth[i],
            centre, activity, mean_p, at_half,
        ]
    return out


# --------------------------------------------------------------------------- labels from ground truth
def node_hits(tree: dict, gt_sets: list) -> list:
    """For every record, per node: the ground-truth instances it matches with IoU > 0.5.

    ``gt_sets`` is ``[(record_id, [RLE, ...]), ...]``. Returns, per record, a dict node -> [(gt_index, iou)].
    Candidates of one selection are disjoint, so each ground truth is hit by at most one of them.
    """
    out = []
    for _, masks in gt_sets:
        decoded = [coco.decode(m).astype(bool) for m in masks]
        boxes = [coco.toBbox(m) for m in masks]
        areas = [float(coco.area(m)) for m in masks]
        hits = {}
        for i, nd in enumerate(tree["nodes"]):
            x, y, w, h = nd["x"], nd["y"], nd["w"], nd["h"]
            mask = None
            for g, (bx, by, bw, bh) in enumerate(boxes):
                if bx > x + w or bx + bw < x or by > y + h or by + bh < y:
                    continue
                if mask is None:
                    mask = node_mask(tree, i)
                inter = float(np.count_nonzero(decoded[g][y:y + h, x:x + w] & mask))
                iou = inter / (nd["area"] + areas[g] - inter) if inter else 0.0
                if iou > 0.5:
                    hits.setdefault(i, []).append((g, iou))
        out.append(hits)
    return out


def hit_targets(n_nodes: int, hits: list) -> tuple:
    """Per node: u = mean over records of summed matched IoU, h = share of records with a match."""
    u, h = np.zeros(n_nodes), np.zeros(n_nodes)
    for record in hits:
        for i, found in record.items():
            u[i] += sum(iou for _, iou in found)
            h[i] += 1.0
    r = max(len(hits), 1)
    return u / r, h / r


def score_selection(bank: dict, selected: list) -> list:
    """count_matches-style rows (tp, fp, fn, sum_iou) per record for the selected node indices."""
    rows = []
    chosen = set(selected)
    for rid, n_gt, hits in zip(bank["record_ids"], bank["n_gt"], bank["hits"]):
        tp, sum_iou, matched, with_hit = 0, 0.0, set(), 0
        for i in chosen:
            found = hits.get(i)
            if found:
                with_hit += 1
                for g, iou in found:
                    tp += 1
                    sum_iou += iou
                    matched.add(g)
        rows.append({"file_name": bank["name"], "image_record": rid, "tp": tp, "fp": len(chosen) - with_hit,
                     "fn": n_gt - len(matched), "sum_iou": sum_iou})
    return rows


# --------------------------------------------------------------------------- scorer
class NodeScorer:
    """Expected matched IoU of a candidate. ``hurdle``: P(match) x E[IoU | match]; ``direct``: regress u."""

    def __init__(self, mode: str = "hurdle", max_iter: int = 250, learning_rate: float = 0.05,
                 max_leaf_nodes: int = 15, min_samples_leaf: int = 40, l2: float = 1.0, seed: int = 0):
        self.mode = mode
        self.params = dict(max_iter=max_iter, learning_rate=learning_rate, max_leaf_nodes=max_leaf_nodes,
                           min_samples_leaf=min_samples_leaf, l2_regularization=l2, random_state=seed,
                           early_stopping=False)
        self.models = {}

    def _regressor(self):
        from sklearn.ensemble import HistGradientBoostingRegressor
        return HistGradientBoostingRegressor(**self.params)

    def fit(self, features: np.ndarray, u: np.ndarray, h: np.ndarray, weight=None) -> "NodeScorer":
        weight = np.ones(len(u)) if weight is None else np.asarray(weight, float)
        if self.mode == "direct":
            self.models = {"u": self._regressor().fit(features, u, sample_weight=weight)}
        else:
            self.models = {"h": self._regressor().fit(features, h, sample_weight=weight)}
            matched = h > 0
            if matched.sum() >= 10:
                self.models["m"] = self._regressor().fit(features[matched], u[matched] / h[matched],
                                                         sample_weight=weight[matched] * h[matched])
            else:
                self.models["m"] = None
        return self

    def predict(self, features: np.ndarray) -> np.ndarray:
        if not len(features):
            return np.zeros(0)
        if self.mode == "direct":
            return np.clip(self.models["u"].predict(features), 0.0, 1.0)
        h = np.clip(self.models["h"].predict(features), 0.0, 1.0)
        m = np.full(len(features), 0.65) if self.models["m"] is None else np.clip(self.models["m"].predict(features), 0.5, 1.0)
        return h * m


# --------------------------------------------------------------------------- exact selection
def select_antichain(parent, gain, selectable) -> list:
    """Indices of disjoint nodes maximising sum(gain). Parents must precede children (parent[i] < i)."""
    n = len(parent)
    child_sum = np.zeros(n)
    choose = np.zeros(n, bool)
    for i in range(n - 1, -1, -1):
        if selectable[i] and gain[i] > 0 and gain[i] > child_sum[i]:
            choose[i], best = True, gain[i]
        else:
            best = child_sum[i]
        if parent[i] >= 0:
            child_sum[parent[i]] += best
    selected, blocked = [], np.zeros(n, bool)   # blocked: the node or an ancestor is already selected
    for i in range(n):
        if parent[i] >= 0 and blocked[parent[i]]:
            blocked[i] = True
        elif choose[i]:
            selected.append(i)
            blocked[i] = True
    return selected


# --------------------------------------------------------------------------- banks and end to end
def build_bank(name: str, prob: np.ndarray, disk: np.ndarray, cfg: TreeConfig, gt_sets=None) -> dict:
    """Tree, features and (when ground truth is given) per-record hits for one image. No label images kept."""
    tree = build_tree(prob, disk, cfg)
    nodes = tree["nodes"]
    bank = {"name": name, "features": node_features(tree, disk),
            "parent": np.array([nd["parent"] for nd in nodes], np.int64),
            "area": np.array([nd["area"] for nd in nodes], np.int64),
            "selectable": np.array([nd["area"] <= cfg.max_area for nd in nodes], bool),
            "level": np.array([nd["level"] for nd in nodes], np.int64)}
    if gt_sets is not None:
        hits = node_hits(tree, gt_sets)
        u, h = hit_targets(len(nodes), hits)
        bank.update(record_ids=[rid for rid, _ in gt_sets], n_gt=[len(m) for _, m in gt_sets], hits=hits,
                    u=u, h=h, n_records=len(gt_sets))
    return bank


def select_instances(prob: np.ndarray, disk: np.ndarray, cfg: TreeConfig, scorer: NodeScorer, tau: float):
    """Probability map -> (label image, accepted labels), as dl.segment returns them."""
    tree = build_tree(prob, disk, cfg)
    nodes = tree["nodes"]
    labels = np.zeros(prob.shape, np.int32)
    if not nodes:
        return labels, []
    features = node_features(tree, disk)
    parent = np.array([nd["parent"] for nd in nodes])
    selectable = np.array([nd["area"] <= cfg.max_area for nd in nodes])
    chosen = select_antichain(parent, scorer.predict(features) - tau, selectable)
    for k, i in enumerate(chosen, 1):
        nd = tree["nodes"][i]
        window = labels[nd["y"]:nd["y"] + nd["h"], nd["x"]:nd["x"] + nd["w"]]
        window[node_mask(tree, i)] = k
    return labels, list(range(1, len(chosen) + 1))


def config_dict(cfg: TreeConfig) -> dict:
    out = asdict(cfg)
    out["levels"] = list(out["levels"])
    return out


# --------------------------------------------------------------------------- experiment orchestration
import json
import os
import pickle
import time
from multiprocessing import get_context
from pathlib import Path

from .core import aggregate

_BANK_STATE: dict = {}
_TEST_STATE: dict = {}


def _bank_job(index: int) -> dict:
    item = _BANK_STATE["items"][index]
    prob = _BANK_STATE["load"](item["name"])
    disk = _BANK_STATE["disk"](item["path"])
    return build_bank(item["name"], prob, disk, _BANK_STATE["cfg"], item.get("gt"))


def build_banks(items: list, load, disk_fn, cfg: TreeConfig, workers: int) -> list:
    """One bank per image, built in a fork pool. ``load(name)`` returns the probability map."""
    _BANK_STATE.update(items=items, load=load, disk=disk_fn, cfg=cfg)
    try:
        if workers > 1:
            with get_context("fork").Pool(workers) as pool:
                return pool.map(_bank_job, range(len(items)), chunksize=1)
        return [_bank_job(i) for i in range(len(items))]
    finally:
        _BANK_STATE.clear()


def _stack(banks: list) -> tuple:
    features = np.vstack([b["features"] for b in banks if len(b["features"])])
    keep = [b for b in banks if len(b["features"])]
    u = np.concatenate([b["u"] for b in keep])
    h = np.concatenate([b["h"] for b in keep])
    w = np.concatenate([np.full(len(b["u"]), float(b["n_records"])) for b in keep])
    return features, u, h, w


def fit_scorer(banks: list, mode: str, **kwargs) -> NodeScorer:
    features, u, h, w = _stack(banks)
    return NodeScorer(mode, **kwargs).fit(features, u, h, w)


def crossfit(banks: list, mode: str, folds: int = 5, **kwargs) -> list:
    """Out-of-fold expected-IoU predictions; whole capture dates stay in one fold."""
    keys = [b["name"][:8] for b in banks]
    if len(set(keys)) < 2:          # fewer than two capture dates (tiny test sets): fall back to single images
        keys = [b["name"] for b in banks]
    dates = sorted(set(keys))
    fold_of = {d: i % folds for i, d in enumerate(dates)}
    out = [None] * len(banks)
    for fold in range(min(folds, len(dates))):
        train = [b for b, k in zip(banks, keys) if fold_of[k] != fold]
        scorer = fit_scorer(train, mode, **kwargs)
        for i, k in enumerate(keys):
            if fold_of[k] == fold:
                out[i] = scorer.predict(banks[i]["features"])
    return out


def evaluate_tau(banks: list, uhat: list, tau: float) -> tuple:
    """Pooled PQ of the exact selection at ``tau`` (rows are returned for bootstrapping)."""
    rows = []
    for bank, u in zip(banks, uhat):
        chosen = select_antichain(bank["parent"], u - tau, bank["selectable"]) if len(u) else []
        rows += score_selection(bank, chosen)
    return aggregate(rows)["pq"], rows


def sweep_tau(banks: list, uhat: list, taus) -> dict:
    return {round(float(t), 4): evaluate_tau(banks, uhat, t)[0] for t in taus}


def _test_bank_job(index: int) -> dict:
    from .dl import solar_disk
    st = _TEST_STATE
    path = st["paths"][index]
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    return build_bank(Path(path).name, st["load"](Path(path).name), solar_disk(img), st["cfg"])


def _test_mask_job(index: int) -> list:
    from .dl import encode_instances_fast, solar_disk
    st = _TEST_STATE
    path = st["paths"][index]
    chosen = st["chosen"][index]
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    tree = build_tree(st["load"](Path(path).name), solar_disk(img), st["cfg"])
    labels = np.zeros(img.shape, np.int32)
    for k, i in enumerate(chosen, 1):
        nd = tree["nodes"][i]
        labels[nd["y"]:nd["y"] + nd["h"], nd["x"]:nd["x"] + nd["w"]][node_mask(tree, i)] = k
    return [(f"{Path(path).stem}_{n}", rle["counts"].decode("ascii"))
            for n, rle in enumerate(encode_instances_fast(labels, list(range(1, len(chosen) + 1))), 1)]


def _map(job, count: int, workers: int) -> list:
    if workers > 1:
        with get_context("fork").Pool(workers) as pool:
            return pool.map(job, range(count), chunksize=2)
    return [job(i) for i in range(count)]


def write_tree_csv(test_paths: list, load, cfg: TreeConfig, scorer: NodeScorer, tau: float, out_csv, workers: int) -> None:
    """Workers never run scikit-learn (its OpenMP threads deadlock after a fork): workers build banks,
    the parent scores and selects, workers cut the chosen masks."""
    paths = [str(p) for p in test_paths]
    _TEST_STATE.update(paths=paths, load=load, cfg=cfg)
    try:
        banks = _map(_test_bank_job, len(paths), workers)
        _TEST_STATE["chosen"] = [
            select_antichain(b["parent"], scorer.predict(b["features"]) - tau, b["selectable"]) if len(b["parent"]) else []
            for b in banks]
        parts = _map(_test_mask_job, len(paths), workers)
    finally:
        _TEST_STATE.clear()
    with Path(out_csv).open("w") as handle:
        handle.write("filament_id,segmentation_rle\n")
        for part in parts:
            for fid, counts in part:
                handle.write(f"{fid},{counts}\n")


def run_treesel(data_dir, split: dict, probs_dir, members: list, out_dir, baseline_post=None, refit=None,
                cfg=None, workers=None, taus=None, debug: bool = False, log=print) -> dict:
    """Tune on calibration (cross-fitted), score validation once, write test CSVs. Nothing is uploaded.

    ``probs_dir``/``members``: cached probability maps of models trained WITHOUT the calibration and
    validation images, for the calibration, validation and test images. ``refit``: optional
    ``{"probs_dir": ..., "members": [...]}`` of models retrained on all labelled images, used for a
    second test CSV (the scorer is then trained on calibration + validation maps of the held-out models).
    """
    from .core import load_annotations
    from .dl import bootstrap_ci, list_images, load_cases, solar_disk, verify_submission
    from .suite import Evaluator, _load_probs

    cv2.setNumThreads(1)
    data_dir, out_dir, probs_dir = Path(data_dir), Path(out_dir), Path(probs_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    cfg = cfg or TreeConfig()
    workers = workers or max(1, min(4, os.cpu_count() or 1))
    taus = list(taus) if taus is not None else [round(0.06 + 0.02 * k, 2) for k in range(20)]
    t0 = time.time()
    tune_names, val_names = list(split["calibration"]), list(split["validation"])
    test_paths = list_images(data_dir / "test/test_images")
    if debug:
        tune_names, val_names, test_paths = tune_names[:6], val_names[:4], test_paths[:3]
    records, annotations = load_annotations(data_dir)
    tune_cases = load_cases(data_dir, tune_names, records, annotations, log)
    val_cases = load_cases(data_dir, val_names, records, annotations, log)
    train_dir = data_dir / "train/train_images"

    def loader(dir_, mem):
        return lambda name: _load_probs(Path(dir_), mem, [name])[name]

    def disk_of(path):
        return solar_disk(cv2.imread(path, cv2.IMREAD_GRAYSCALE))

    def items(cases):
        return [{"name": c["name"], "path": str(train_dir / c["name"]), "gt": c["gt"]} for c in cases]

    held_out = loader(probs_dir, members)
    log(f"treesel: building banks for {len(tune_cases)} calibration + {len(val_cases)} validation images")
    banks_cal = build_banks(items(tune_cases), held_out, disk_of, cfg, workers)
    banks_val = build_banks(items(val_cases), held_out, disk_of, cfg, workers)
    nodes_cal = sum(len(b["parent"]) for b in banks_cal)
    log(f"  nodes: calibration {nodes_cal}, validation {sum(len(b['parent']) for b in banks_val)} "
        f"({time.time() - t0:.0f}s)")

    # oracle: the best any scorer could do with this tree (true expected IoU as the score)
    oracle = {}
    for name, banks in (("calibration", banks_cal), ("validation", banks_val)):
        sweep = sweep_tau(banks, [b["u"] for b in banks], taus)
        oracle[name] = {"best_pq": max(sweep.values()), "best_tau": max(sweep, key=sweep.get)}
    log(f"  oracle (true u as the score): {oracle}")

    # choose scorer mode and tau on calibration only, with out-of-fold scores
    tuning = {}
    for mode in ("hurdle", "direct"):
        uhat = crossfit(banks_cal, mode)
        sweep = sweep_tau(banks_cal, uhat, taus)
        tuning[mode] = {"sweep": sweep, "best_tau": max(sweep, key=sweep.get), "best_pq": max(sweep.values())}
        log(f"  {mode}: calibration out-of-fold PQ {tuning[mode]['best_pq']:.4f} at tau {tuning[mode]['best_tau']}")
    mode = max(tuning, key=lambda m: tuning[m]["best_pq"])
    # smooth the tau choice: the centre of the taus within 0.002 PQ of the best
    sweep = tuning[mode]["sweep"]
    good = [t for t, v in sweep.items() if v >= tuning[mode]["best_pq"] - 0.002]
    tau = float(np.median(good))
    log(f"  chosen on calibration: mode={mode} tau={tau:.3f} (within 0.002 of best: {good})")

    # validation, scored once
    scorer = fit_scorer(banks_cal, mode)
    uhat_val = [scorer.predict(b["features"]) for b in banks_val]
    pq_val, rows = evaluate_tau(banks_val, uhat_val, tau)
    summary = aggregate(rows)
    interval, clusters = bootstrap_ci(rows)
    summary.update(pq_bootstrap_95ci=interval, date_clusters=clusters, annotation_sets=len(rows))
    sensitivity = sweep_tau(banks_val, uhat_val, taus)   # reported, never used to choose
    log(f"  VALIDATION pq={summary['pq']:.4f} CI {interval[0]:.4f}-{interval[1]:.4f} "
        f"tp/fp/fn={summary['tp']}/{summary['fp']}/{summary['fn']}")

    baseline = None
    if baseline_post is not None:
        evaluator = Evaluator(val_cases, _load_probs(probs_dir, members, val_names), workers)
        try:
            baseline = aggregate(evaluator.rows(baseline_post))
        finally:
            evaluator.close()
        log(f"  fixed-rule baseline on the same maps: validation pq={baseline['pq']:.4f}")

    # final scorer: every labelled image whose maps come from held-out models
    final_scorer = fit_scorer(banks_cal + banks_val, mode)
    with (out_dir / "scorer.pkl").open("wb") as handle:
        pickle.dump({"scorer": final_scorer, "tau": tau, "cfg": config_dict(cfg), "mode": mode}, handle)
    outputs = {}
    plans = [("held_out_models", probs_dir, members)]
    if refit:
        plans.append(("refit_models", Path(refit["probs_dir"]), refit["members"]))
    for label, dir_, mem in plans:
        csv_path = out_dir / f"submission_{label}.csv"
        write_tree_csv(test_paths, loader(dir_, mem), cfg, final_scorer, tau, csv_path, workers)
        outputs[label] = verify_submission(csv_path, data_dir / "test/test_images")
        log(f"  {label}: {outputs[label]}")

    metrics = {"mode": mode, "tau": tau, "tree_config": config_dict(cfg), "calibration_tuning": tuning,
               "oracle": oracle, "validation": summary, "validation_tau_sensitivity": sensitivity,
               "fixed_rule_baseline_validation": baseline, "csv": outputs, "members": members,
               "nodes_calibration": nodes_cal, "minutes": (time.time() - t0) / 60, "debug": debug}
    (out_dir / "treesel_metrics.json").write_text(json.dumps(metrics, indent=2, default=float))
    return metrics


In [ ]:
sys.path.insert(0, str(Path('code').resolve()))
from filament import suite, treesel
from filament.dl import PostConfig, list_images
print('modules:', suite.__file__, treesel.__file__)

In [ ]:
SPLIT = json.loads(r'''{"seed":2026,"grouping":"capture date, stratified by year","validation":["20110119082634Lh.jpeg","20110123174914Mh.jpeg","20110203082634Lh.jpeg","20110215083814Th.jpeg","20110312082634Lh.jpeg","20110325171014Mh.jpeg","20110406082634Lh.jpeg","20110511063154Uh.jpeg","20110526122534Ch.jpeg","20110615133454Bh.jpeg","20110616133454Bh.jpeg","20110810063134Lh.jpeg","20111018063134Lh.jpeg","20111019063154Uh.jpeg","20111029063154Uh.jpeg","20111215063134Lh.jpeg","20111222063154Uh.jpeg","20120217063154Uh.jpeg","20120217181154Bh.jpeg","20120227161654Bh.jpeg","20120318063154Uh.jpeg","20120331142954Bh.jpeg","20120421063154Uh.jpeg","20120509163414Mh.jpeg","20120523163014Mh.jpeg","20120610175114Mh.jpeg","20120715123434Ch.jpeg","20120723095014Th.jpeg","20120729122634Ch.jpeg","20120730135454Bh.jpeg","20120811121434Ch.jpeg","20120901114934Ch.jpeg","20120907141954Bh.jpeg","20121129063154Uh.jpeg","20130116172814Mh.jpeg","20130221063134Lh.jpeg","20130304170614Mh.jpeg","20130404063134Lh.jpeg","20130429063154Uh.jpeg","20130716162214Mh.jpeg","20130725122934Ch.jpeg","20130805135854Bh.jpeg","20130807073114Th.jpeg","20130808140054Bh.jpeg","20130913142354Bh.jpeg","20130925143254Bh.jpeg","20131020080114Th.jpeg","20131028190734Ch.jpeg","20131030185354Bh.jpeg","20131108045254Uh.jpeg","20131201192134Ch.jpeg","20131204192154Bh.jpeg","20131229112414Th.jpeg","20140116180214Mh.jpeg","20140127180134Ch.jpeg","20140128174554Bh.jpeg","20140205180154Bh.jpeg","20140409081614Th.jpeg","20140523195814Mh.jpeg","20140525070514Th.jpeg","20140604195854Bh.jpeg","20140607112354Uh.jpeg","20140607195854Bh.jpeg","20140615195814Mh.jpeg","20140615195854Bh.jpeg","20140817195834Ch.jpeg","20140821195834Ch.jpeg","20141007224934Lh.jpeg","20141010195834Ch.jpeg","20141025031154Uh.jpeg","20141214201834Ch.jpeg","20141216222634Lh.jpeg","20141225024354Uh.jpeg","20141225172214Mh.jpeg","20150112081954Uh.jpeg","20150113105034Ch.jpeg","20150113155654Bh.jpeg","20150125172714Mh.jpeg","20150222090014Th.jpeg","20150222112034Ch.jpeg","20150225171114Mh.jpeg","20150324145954Bh.jpeg","20150527082014Th.jpeg","20150527161214Mh.jpeg","20150610082014Th.jpeg","20150615133554Bh.jpeg","20150630083214Th.jpeg","20150706083014Th.jpeg","20150916055454Uh.jpeg","20151106171514Mh.jpeg","20151118172314Mh.jpeg","20151119133734Ch.jpeg","20151209201754Bh.jpeg","20160229151254Bh.jpeg","20160302133734Ch.jpeg","20160309172214Mh.jpeg","20160317232434Lh.jpeg","20160320133714Th.jpeg","20160326143654Bh.jpeg","20160327170714Mh.jpeg","20160518011654Uh.jpeg","20160724135554Bh.jpeg","20160804133734Ch.jpeg","20160920230134Lh.jpeg","20161012133734Ch.jpeg","20161023133814Th.jpeg","20161110151454Bh.jpeg","20161210133714Th.jpeg","20170217165250Bh.jpeg","20170323041730Lh.jpeg","20170417014150Uh.jpeg","20170417232530Lh.jpeg","20170418013450Uh.jpeg","20170427154550Bh.jpeg","20170619012150Uh.jpeg","20170722235230Lh.jpeg","20170805133710Th.jpeg","20170927143450Bh.jpeg","20171004143950Bh.jpeg","20171014133730Ch.jpeg","20171027223030Lh.jpeg","20171121222130Lh.jpeg","20180309000910Mh.jpeg","20180801232450Bh.jpeg","20180825205030Ch.jpeg","20180925000030Lh.jpeg","20181102104630Ch.jpeg","20190609123330Ch.jpeg","20190626025430Lh.jpeg","20190701123830Ch.jpeg","20200110073230Lh.jpeg","20200619123010Th.jpeg","20210103223630Lh.jpeg","20210201020330Lh.jpeg","20210616142650Bh.jpeg","20210817165030Mh.jpeg","20211231223430Lh.jpeg","20220617185352Mh.jpeg","20220803074752Th.jpeg"],"calibration":["20110109104734Ch.jpeg","20110220082634Lh.jpeg","20110221173814Mh.jpeg","20110301082654Uh.jpeg","20110313082654Uh.jpeg","20110324082614Th.jpeg","20110327143554Bh.jpeg","20110421164914Mh.jpeg","20110509121334Ch.jpeg","20110711071214Th.jpeg","20110727063234Lh.jpeg","20111116063134Lh.jpeg","20111223085314Th.jpeg","20120127085214Th.jpeg","20120205172414Mh.jpeg","20120213063134Lh.jpeg","20120223063154Uh.jpeg","20120223152054Bh.jpeg","20120225172914Mh.jpeg","20120414115434Ch.jpeg","20120414175914Mh.jpeg","20120816072814Th.jpeg","20120928063134Lh.jpeg","20121108155954Bh.jpeg","20121114172254Bh.jpeg","20121118092514Th.jpeg","20121218063534Lh.jpeg","20130214111534Ch.jpeg","20130219063134Lh.jpeg","20130303063134Lh.jpeg","20130303063154Uh.jpeg","20130324015554Uh.jpeg","20130326063134Lh.jpeg","20130517063134Lh.jpeg","20130609123334Ch.jpeg","20130609173414Mh.jpeg","20130616070414Th.jpeg","20130903060834Lh.jpeg","20130903141654Bh.jpeg","20130905073714Th.jpeg","20130926060834Lh.jpeg","20131019080114Th.jpeg","20131025185414Mh.jpeg","20131031185614Mh.jpeg","20140212024554Uh.jpeg","20140225163314Th.jpeg","20140312025454Uh.jpeg","20140312081114Th.jpeg","20140414020354Uh.jpeg","20140414195834Ch.jpeg","20140710083214Th.jpeg","20140725195814Mh.jpeg","20140802072114Th.jpeg","20140816072814Th.jpeg","20140818195814Mh.jpeg","20141012015654Uh.jpeg","20141012195854Bh.jpeg","20141013195834Ch.jpeg","20141013195854Bh.jpeg","20141016022554Uh.jpeg","20141016195834Ch.jpeg","20141026024454Uh.jpeg","20141105195814Mh.jpeg","20150216171714Mh.jpeg","20150325015554Uh.jpeg","20150427162314Mh.jpeg","20150627133754Bh.jpeg","20150719182314Mh.jpeg","20150731082034Lh.jpeg","20150819082014Th.jpeg","20151011144554Bh.jpeg","20151105133734Ch.jpeg","20151107133734Ch.jpeg","20151224133734Ch.jpeg","20151226133734Ch.jpeg","20160103133734Ch.jpeg","20160110133714Th.jpeg","20160310133734Ch.jpeg","20160430133714Th.jpeg","20160430133734Ch.jpeg","20160516012454Uh.jpeg","20160530234734Lh.jpeg","20160604163414Mh.jpeg","20160703133714Th.jpeg","20160726135354Bh.jpeg","20160808133714Th.jpeg","20161223133734Ch.jpeg","20170503013050Uh.jpeg","20170612135150Bh.jpeg","20170801164710Mh.jpeg","20170804140650Bh.jpeg","20170809164910Mh.jpeg","20170831170750Bh.jpeg","20171008144250Bh.jpeg","20171022170710Mh.jpeg","20171025133730Ch.jpeg","20171025223130Lh.jpeg","20180112130330Ch.jpeg","20180805232510Mh.jpeg","20180923112330Ch.jpeg","20180924112030Ch.jpeg","20190611032810Mh.jpeg","20190926165810Mh.jpeg","20200806053830Lh.jpeg","20210329015250Uh.jpeg","20210329114430Ch.jpeg","20210331231510Mh.jpeg","20210620163530Mh.jpeg","20210815140550Bh.jpeg","20220104043050Uh.jpeg","20220618185332Bh.jpeg"],"training":["20110109104734Ch.jpeg","20110114105034Ch.jpeg","20110128174814Mh.jpeg","20110205082634Lh.jpeg","20110211084114Th.jpeg","20110213093414Th.jpeg","20110220082634Lh.jpeg","20110221173814Mh.jpeg","20110301082654Uh.jpeg","20110306082634Lh.jpeg","20110313082654Uh.jpeg","20110317082654Uh.jpeg","20110322005814Mh.jpeg","20110324082614Th.jpeg","20110326170914Mh.jpeg","20110327143554Bh.jpeg","20110410141654Bh.jpeg","20110421164914Mh.jpeg","20110425063154Uh.jpeg","20110429135554Bh.jpeg","20110502063154Uh.jpeg","20110509121334Ch.jpeg","20110510063134Lh.jpeg","20110517164514Mh.jpeg","20110520063154Uh.jpeg","20110521063134Lh.jpeg","20110523063154Uh.jpeg","20110525063154Uh.jpeg","20110530122834Ch.jpeg","20110601063134Lh.jpeg","20110603163414Mh.jpeg","20110617063134Lh.jpeg","20110619063134Lh.jpeg","20110621063134Lh.jpeg","20110623063134Lh.jpeg","20110625163814Mh.jpeg","20110707074514Th.jpeg","20110711071214Th.jpeg","20110724164614Mh.jpeg","20110727063234Lh.jpeg","20110728084714Th.jpeg","20110803164914Mh.jpeg","20110815140454Bh.jpeg","20110823120334Ch.jpeg","20110824141054Bh.jpeg","20110826074214Th.jpeg","20110902073614Th.jpeg","20110914063134Lh.jpeg","20110918112834Ch.jpeg","20110923112234Ch.jpeg","20110923165814Mh.jpeg","20111009110234Ch.jpeg","20111011112054Uh.jpeg","20111025080614Th.jpeg","20111027145954Bh.jpeg","20111107063154Uh.jpeg","20111112193254Bh.jpeg","20111114082314Th.jpeg","20111115063134Lh.jpeg","20111116063134Lh.jpeg","20111201063154Uh.jpeg","20111206102934Ch.jpeg","20111208063134Lh.jpeg","20111214063134Lh.jpeg","20111214132434Ch.jpeg","20111218085014Th.jpeg","20111223085314Th.jpeg","20111226155454Bh.jpeg","20111229063154Uh.jpeg","20120103104134Ch.jpeg","20120104063134Lh.jpeg","20120110155754Bh.jpeg","20120113104834Ch.jpeg","20120114063154Uh.jpeg","20120116063254Uh.jpeg","20120127085214Th.jpeg","20120131063154Uh.jpeg","20120205172414Mh.jpeg","20120206063134Lh.jpeg","20120213063134Lh.jpeg","20120215153654Bh.jpeg","20120218173614Mh.jpeg","20120223063154Uh.jpeg","20120223152054Bh.jpeg","20120225172914Mh.jpeg","20120306063154Uh.jpeg","20120313063154Uh.jpeg","20120323144054Bh.jpeg","20120328164614Mh.jpeg","20120330063134Lh.jpeg","20120330063154Uh.jpeg","20120403083114Th.jpeg","20120403164114Mh.jpeg","20120404063154Uh.jpeg","20120414115434Ch.jpeg","20120414175914Mh.jpeg","20120424072414Th.jpeg","20120424120134Ch.jpeg","20120427063154Uh.jpeg","20120427162614Mh.jpeg","20120430063154Uh.jpeg","20120519070814Th.jpeg","20120603123034Ch.jpeg","20120627162114Mh.jpeg","20120703070814Th.jpeg","20120712123634Ch.jpeg","20120720085114Th.jpeg","20120725122934Ch.jpeg","20120726071914Th.jpeg","20120802090314Th.jpeg","20120803102114Th.jpeg","20120803164614Mh.jpeg","20120810121634Ch.jpeg","20120813164714Mh.jpeg","20120814140454Bh.jpeg","20120814163114Mh.jpeg","20120816072814Th.jpeg","20120827085114Th.jpeg","20120829063134Lh.jpeg","20120910074014Th.jpeg","20120913165314Mh.jpeg","20120914063134Lh.jpeg","20120917142654Bh.jpeg","20120928063134Lh.jpeg","20121006063534Lh.jpeg","20121007131134Ch.jpeg","20121009165914Mh.jpeg","20121018063134Lh.jpeg","20121021145454Bh.jpeg","20121022145554Bh.jpeg","20121024164714Mh.jpeg","20121026063134Lh.jpeg","20121027063134Lh.jpeg","20121027165114Mh.jpeg","20121030150354Bh.jpeg","20121108155954Bh.jpeg","20121109103634Ch.jpeg","20121113103634Ch.jpeg","20121114172254Bh.jpeg","20121118092514Th.jpeg","20121207103034Ch.jpeg","20121208171514Mh.jpeg","20121218063534Lh.jpeg","20121219063154Uh.jpeg","20130107174854Bh.jpeg","20130109130334Ch.jpeg","20130110104734Ch.jpeg","20130111104734Ch.jpeg","20130201084914Th.jpeg","20130208063154Uh.jpeg","20130211063134Lh.jpeg","20130212153254Bh.jpeg","20130214111534Ch.jpeg","20130218220314Mh.jpeg","20130219063134Lh.jpeg","20130222082914Th.jpeg","20130225171214Mh.jpeg","20130303063134Lh.jpeg","20130303063154Uh.jpeg","20130305112734Ch.jpeg","20130306172214Mh.jpeg","20130314165714Mh.jpeg","20130319144654Bh.jpeg","20130324015554Uh.jpeg","20130326063134Lh.jpeg","20130327175114Mh.jpeg","20130405114734Ch.jpeg","20130407063154Uh.jpeg","20130411115134Ch.jpeg","20130419070254Uh.jpeg","20130423072414Th.jpeg","20130425135954Bh.jpeg","20130502063134Lh.jpeg","20130510134654Bh.jpeg","20130517063134Lh.jpeg","20130601063154Uh.jpeg","20130609123334Ch.jpeg","20130609173414Mh.jpeg","20130611063154Uh.jpeg","20130615002434Lh.jpeg","20130616070414Th.jpeg","20130628063134Lh.jpeg","20130630070714Th.jpeg","20130705161714Mh.jpeg","20130707134154Bh.jpeg","20130717123434Ch.jpeg","20130719140154Bh.jpeg","20130809072814Th.jpeg","20130810121534Ch.jpeg","20130811140254Bh.jpeg","20130813060834Lh.jpeg","20130824060834Lh.jpeg","20130828141354Bh.jpeg","20130901073614Th.jpeg","20130903060834Lh.jpeg","20130903141654Bh.jpeg","20130905073714Th.jpeg","20130919060834Lh.jpeg","20130921060834Lh.jpeg","20130923060834Lh.jpeg","20130926060834Lh.jpeg","20131006075314Th.jpeg","20131007075314Th.jpeg","20131011195414Mh.jpeg","20131013185334Ch.jpeg","20131016185334Ch.jpeg","20131017185334Ch.jpeg","20131019080114Th.jpeg","20131025185414Mh.jpeg","20131027025654Uh.jpeg","20131031185614Mh.jpeg","20131101222734Lh.jpeg","20131107043054Uh.jpeg","20131109190514Mh.jpeg","20131109222434Lh.jpeg","20131113004514Mh.jpeg","20131113185414Mh.jpeg","20131115185414Mh.jpeg","20131116185334Ch.jpeg","20131125192154Bh.jpeg","20131128192214Mh.jpeg","20131130192134Ch.jpeg","20131206084114Th.jpeg","20131207084214Th.jpeg","20131224192154Bh.jpeg","20131225065154Uh.jpeg","20140111224134Lh.jpeg","20140112180134Ch.jpeg","20140122180214Mh.jpeg","20140210180154Bh.jpeg","20140212024554Uh.jpeg","20140213024554Uh.jpeg","20140216230134Lh.jpeg","20140218230234Lh.jpeg","20140225163314Th.jpeg","20140312025454Uh.jpeg","20140312081114Th.jpeg","20140321195814Mh.jpeg","20140323020654Uh.jpeg","20140401014754Uh.jpeg","20140406195854Bh.jpeg","20140411013854Uh.jpeg","20140413195914Mh.jpeg","20140414020354Uh.jpeg","20140414195834Ch.jpeg","20140420085614Th.jpeg","20140428233234Lh.jpeg","20140430071914Th.jpeg","20140501195814Mh.jpeg","20140508195834Ch.jpeg","20140509071314Th.jpeg","20140509195814Mh.jpeg","20140513012554Uh.jpeg","20140522070614Th.jpeg","20140531195834Ch.jpeg","20140601195814Mh.jpeg","20140603070414Th.jpeg","20140605195854Bh.jpeg","20140609195854Bh.jpeg","20140701195814Mh.jpeg","20140705035754Uh.jpeg","20140709195814Mh.jpeg","20140710083214Th.jpeg","20140712195854Bh.jpeg","20140716195854Bh.jpeg","20140719071414Th.jpeg","20140720195834Ch.jpeg","20140723071614Th.jpeg","20140725195814Mh.jpeg","20140726071814Th.jpeg","20140802072114Th.jpeg","20140803234534Lh.jpeg","20140806195834Ch.jpeg","20140807195854Bh.jpeg","20140813064254Uh.jpeg","20140814195854Bh.jpeg","20140816072814Th.jpeg","20140818195814Mh.jpeg","20140903073614Th.jpeg","20140904195814Mh.jpeg","20140908175354Bh.jpeg","20140908231334Lh.jpeg","20140918195854Bh.jpeg","20140921195854Bh.jpeg","20140922195854Bh.jpeg","20140927014554Uh.jpeg","20141001195834Ch.jpeg","20141004075414Th.jpeg","20141006224634Lh.jpeg","20141009023454Uh.jpeg","20141012015654Uh.jpeg","20141012195854Bh.jpeg","20141013195834Ch.jpeg","20141013195854Bh.jpeg","20141016022554Uh.jpeg","20141016195834Ch.jpeg","20141022223334Lh.jpeg","20141024195834Ch.jpeg","20141026024454Uh.jpeg","20141027025054Uh.jpeg","20141027224834Lh.jpeg","20141105195814Mh.jpeg","20141106195854Bh.jpeg","20141112195854Bh.jpeg","20141116222234Lh.jpeg","20141129222134Lh.jpeg","20141210201814Mh.jpeg","20150116082034Lh.jpeg","20150119082034Lh.jpeg","20150130084914Th.jpeg","20150203084714Th.jpeg","20150207084414Th.jpeg","20150216171714Mh.jpeg","20150221155154Bh.jpeg","20150228082414Th.jpeg","20150302082034Lh.jpeg","20150305081954Uh.jpeg","20150308081954Uh.jpeg","20150314081954Uh.jpeg","20150315082034Lh.jpeg","20150323172814Mh.jpeg","20150325015554Uh.jpeg","20150327145114Th.jpeg","20150330081954Uh.jpeg","20150409081954Uh.jpeg","20150417082314Th.jpeg","20150421140454Bh.jpeg","20150423120034Ch.jpeg","20150427162314Mh.jpeg","20150501120634Ch.jpeg","20150518145654Bh.jpeg","20150531082034Lh.jpeg","20150614162514Mh.jpeg","20150617161314Mh.jpeg","20150620082014Th.jpeg","20150621161314Mh.jpeg","20150625082014Th.jpeg","20150625082034Lh.jpeg","20150626161514Mh.jpeg","20150627133754Bh.jpeg","20150629161614Mh.jpeg","20150716162114Mh.jpeg","20150718082014Th.jpeg","20150719182314Mh.jpeg","20150720082014Th.jpeg","20150727082014Th.jpeg","20150731082034Lh.jpeg","20150806082034Lh.jpeg","20150819082014Th.jpeg","20150820152254Bh.jpeg","20150825082034Lh.jpeg","20150830084134Lh.jpeg","20150913082014Th.jpeg","20150919165814Mh.jpeg","20150926111934Ch.jpeg","20150927111834Ch.jpeg","20151011144554Bh.jpeg","20151013144654Bh.jpeg","20151019223534Lh.jpeg","20151105133734Ch.jpeg","20151107133734Ch.jpeg","20151116201114Mh.jpeg","20151116224034Lh.jpeg","20151117172314Mh.jpeg","20151122172614Mh.jpeg","20151128133714Th.jpeg","20151128133734Ch.jpeg","20151128222134Lh.jpeg","20151212171154Bh.jpeg","20151213023454Uh.jpeg","20151219133734Ch.jpeg","20151224133734Ch.jpeg","20151226133734Ch.jpeg","20151228024254Uh.jpeg","20160101133714Th.jpeg","20160103133734Ch.jpeg","20160107133714Th.jpeg","20160110133714Th.jpeg","20160125154854Bh.jpeg","20160127225034Lh.jpeg","20160129225334Lh.jpeg","20160201025254Uh.jpeg","20160204133734Ch.jpeg","20160213153054Bh.jpeg","20160215133734Ch.jpeg","20160215175514Mh.jpeg","20160223173514Mh.jpeg","20160225173314Mh.jpeg","20160226173214Mh.jpeg","20160310133734Ch.jpeg","20160315133714Th.jpeg","20160401133734Ch.jpeg","20160401170314Mh.jpeg","20160402025254Uh.jpeg","20160403170114Mh.jpeg","20160408232234Lh.jpeg","20160411134234Ch.jpeg","20160430133714Th.jpeg","20160430133734Ch.jpeg","20160501143154Bh.jpeg","20160510163914Mh.jpeg","20160516012454Uh.jpeg","20160526011254Uh.jpeg","20160530234734Lh.jpeg","20160604163414Mh.jpeg","20160606163514Mh.jpeg","20160609133654Bh.jpeg","20160617133654Bh.jpeg","20160619133714Th.jpeg","20160627133734Ch.jpeg","20160628152654Bh.jpeg","20160629133714Th.jpeg","20160703133714Th.jpeg","20160714235434Lh.jpeg","20160720133734Ch.jpeg","20160726135354Bh.jpeg","20160808133714Th.jpeg","20160809164914Mh.jpeg","20160810234034Lh.jpeg","20160831133714Th.jpeg","20160925165814Mh.jpeg","20160925225634Lh.jpeg","20160927133734Ch.jpeg","20160928143554Bh.jpeg","20161002225034Lh.jpeg","20161013133734Ch.jpeg","20161105222634Lh.jpeg","20161108021454Uh.jpeg","20161109171714Mh.jpeg","20161116133714Th.jpeg","20161129022454Uh.jpeg","20161201160954Bh.jpeg","20161205133714Th.jpeg","20161217133734Ch.jpeg","20161222133714Th.jpeg","20161223133734Ch.jpeg","20161224024154Uh.jpeg","20161229133734Ch.jpeg","20161231133734Ch.jpeg","20170102024454Uh.jpeg","20170103030554Uh.jpeg","20170106133714Th.jpeg","20170107133714Th.jpeg","20170117175014Mh.jpeg","20170119031254Uh.jpeg","20170122034654Uh.jpeg","20170125133734Ch.jpeg","20170218173710Mh.jpeg","20170222133710Th.jpeg","20170310133730Ch.jpeg","20170310173450Bh.jpeg","20170316133730Ch.jpeg","20170322015550Uh.jpeg","20170327143550Bh.jpeg","20170329133730Ch.jpeg","20170408133710Th.jpeg","20170411013550Uh.jpeg","20170415145450Bh.jpeg","20170423014350Uh.jpeg","20170425133730Ch.jpeg","20170502233130Lh.jpeg","20170503013050Uh.jpeg","20170609133710Th.jpeg","20170612135150Bh.jpeg","20170701235630Lh.jpeg","20170702133730Ch.jpeg","20170719235330Lh.jpeg","20170726235030Lh.jpeg","20170728150230Ch.jpeg","20170801164710Mh.jpeg","20170804140650Bh.jpeg","20170806133710Th.jpeg","20170809164910Mh.jpeg","20170827165310Mh.jpeg","20170831170750Bh.jpeg","20170925225630Lh.jpeg","20171005133730Ch.jpeg","20171005144050Bh.jpeg","20171007133710Th.jpeg","20171007133730Ch.jpeg","20171008144250Bh.jpeg","20171011170310Mh.jpeg","20171011224130Lh.jpeg","20171022170710Mh.jpeg","20171025133730Ch.jpeg","20171025223130Lh.jpeg","20171029150150Bh.jpeg","20171101133730Ch.jpeg","20171103222630Lh.jpeg","20171106171510Mh.jpeg","20171109023950Uh.jpeg","20171114133730Ch.jpeg","20171115170950Bh.jpeg","20171115222230Lh.jpeg","20171123133730Ch.jpeg","20171123222130Lh.jpeg","20180109223930Lh.jpeg","20180112130330Ch.jpeg","20180202130330Ch.jpeg","20180301050430Lh.jpeg","20180306081810Th.jpeg","20180318020350Uh.jpeg","20180318232230Lh.jpeg","20180506140230Ch.jpeg","20180710031950Uh.jpeg","20180712123630Ch.jpeg","20180714000010Mh.jpeg","20180804234530Lh.jpeg","20180805232510Mh.jpeg","20180827232530Lh.jpeg","20180923112330Ch.jpeg","20180924112030Ch.jpeg","20180926014350Uh.jpeg","20181014105830Ch.jpeg","20181023232430Lh.jpeg","20181024232430Lh.jpeg","20181101230010Mh.jpeg","20190317145150Bh.jpeg","20190322144450Bh.jpeg","20190412025430Lh.jpeg","20190602025430Lh.jpeg","20190611032810Mh.jpeg","20190629163710Mh.jpeg","20190704123830Ch.jpeg","20190805054730Lh.jpeg","20190925165810Mh.jpeg","20190926165810Mh.jpeg","20190927165910Mh.jpeg","20200105073250Uh.jpeg","20200114073230Lh.jpeg","20200116073250Uh.jpeg","20200806053830Lh.jpeg","20201022223330Lh.jpeg","20201227202710Mh.jpeg","20201228202430Ch.jpeg","20210222222850Bh.jpeg","20210223222830Ch.jpeg","20210329015250Uh.jpeg","20210329114430Ch.jpeg","20210331231510Mh.jpeg","20210620163530Mh.jpeg","20210621133650Bh.jpeg","20210624133950Bh.jpeg","20210626060050Uh.jpeg","20210626133950Bh.jpeg","20210721123430Ch.jpeg","20210728164530Mh.jpeg","20210731074710Th.jpeg","20210810121630Ch.jpeg","20210812044930Lh.jpeg","20210814044930Lh.jpeg","20210815140550Bh.jpeg","20211016145050Bh.jpeg","20211119103130Ch.jpeg","20211222120130Ch.jpeg","20211225025850Uh.jpeg","20211229120130Ch.jpeg","20211229120210Th.jpeg","20220104043050Uh.jpeg","20220122085210Th.jpeg","20220328143550Bh.jpeg","20220418020950Uh.jpeg","20220618185332Bh.jpeg","20220622205632Bh.jpeg","20220710085152Th.jpeg","20220718185352Mh.jpeg","20220721185332Bh.jpeg","20220724185352Mh.jpeg"]}''')
print({key: len(value) for key, value in SPLIT.items() if isinstance(value, list)})

In [ ]:
annotation_files = list(INPUT_ROOT.rglob('MAGFiLO_1.0_Annotations_kaggle2026_train.json'))
assert len(annotation_files) == 1, f'Attach the competition data; found {len(annotation_files)} annotation files'
DATA = annotation_files[0].parent.parent
v2_files = sorted(INPUT_ROOT.rglob('v2/suite_metrics.json'))
assert v2_files, 'Attach the output of the v2 suite notebook'
V2_DIR = v2_files[0].parent
V2 = json.loads(v2_files[0].read_text())
SELECTED = V2['selected_by_calibration']
MEMBERS = V2['candidates'][SELECTED]['members']
POST = V2['candidates'][SELECTED]['post']
MODELS = [spec for spec in V2['suite']['models'] if spec['name'] in MEMBERS]
assert len(MODELS) == len(MEMBERS)
held_out = [{'name': s['name'], 'encoder': s['cfg']['encoder'], 'cost': s.get('cost', 1.0),
             'weights': str(V2_DIR / 'models' / s['name'] / 'best.pt')} for s in MODELS]
refit_files = sorted(INPUT_ROOT.rglob('refit/refit_metrics.json'))
refit = []
if refit_files:
    refit = [{'name': s['name'], 'encoder': s['cfg']['encoder'], 'cost': s.get('cost', 1.0),
              'weights': str(refit_files[0].parent / 'models' / s['name'] / 'last.pt')} for s in MODELS]
for spec in held_out + refit:
    assert Path(spec['weights']).exists(), spec['weights']
print('data root:', DATA, '| members:', MEMBERS, '| refit models found:', bool(refit))
print('fixed-rule post-processing (v2, calibration-tuned):', POST)

## 1. Probability maps from the saved models

In [ ]:
# probability maps (8-bit PNG, flip TTA) for calibration + validation + test with the held-out models,
# and for test only with the 707-image refit models
train_dir = DATA / 'train/train_images'
test_paths = list_images(DATA / 'test/test_images')
label_names = list(SPLIT['calibration']) + list(SPLIT['validation'])
held_images = {Path(n).stem: str(train_dir / n) for n in label_names}
held_images.update({p.stem: str(p) for p in test_paths})
PROBS = WORK_ROOT / 'probs'
work = WORK_ROOT / 'treesel_work'
work.mkdir(exist_ok=True)
t0 = time.time()
suite._wait(suite._launch('infer', held_out, {'probs': str(PROBS / 'held_out'), 'tta': True, 'images': held_images}, work))
if refit:
    suite._wait(suite._launch('infer', refit, {'probs': str(PROBS / 'refit'), 'tta': True,
                                               'images': {p.stem: str(p) for p in test_paths}}, work))
print(f'inference finished in {(time.time() - t0) / 60:.1f} min')

## 2. Debug pass (minutes)

In [ ]:
# every stage on a handful of images (uses the maps above)
dbg = treesel.run_treesel(DATA, SPLIT, PROBS / 'held_out', MEMBERS, WORK_ROOT / 'treesel_debug',
                          baseline_post=PostConfig(**POST), debug=True)
print({k: dbg[k] for k in ('mode', 'tau', 'csv')})

## 3. Full run: tune on calibration, score validation once, write test CSVs

In [ ]:
res = treesel.run_treesel(DATA, SPLIT, PROBS / 'held_out', MEMBERS, WORK_ROOT / 'treesel',
                          baseline_post=PostConfig(**POST),
                          refit=({'probs_dir': str(PROBS / 'refit'), 'members': MEMBERS} if refit else None))

In [ ]:
v = res['validation']
print(f"scorer mode {res['mode']}, tau {res['tau']:.3f}")
print(f"VALIDATION tree selection : PQ {v['pq']:.4f}  CI {v['pq_bootstrap_95ci']}  TP/FP/FN {v['tp']}/{v['fp']}/{v['fn']}")
b = res['fixed_rule_baseline_validation']
print(f"VALIDATION fixed rule     : PQ {b['pq']:.4f}  TP/FP/FN {b['tp']}/{b['fp']}/{b['fn']}   (v2 reported 0.4271)")
print('oracle ceilings (true expected IoU as score):', res['oracle'])
for m in ('hurdle', 'direct'):
    print(m, 'calibration out-of-fold PQ', round(res['calibration_tuning'][m]['best_pq'], 4), 'tau', res['calibration_tuning'][m]['best_tau'])
print('validation PQ by tau (reported, not used to choose):', {k: round(x, 4) for k, x in res['validation_tau_sensitivity'].items()})
print('test CSVs:', json.dumps(res['csv'], indent=1))